# [1.1] - Transformers from scratch (exercises)

> **ARENA [Streamlit Page](https://arena-chapter1-transformer-interp.streamlit.app/01_[1.1]_Transformer_from_Scratch)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part1_transformer_from_scratch/1.1_Transformer_from_Scratch_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter1_transformer_interp/exercises/part1_transformer_from_scratch/1.1_Transformer_from_Scratch_solutions.ipynb?t=20260329)**

문제나 버그가 있다면 [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA)의 `#errata` 채널로 보내주시고, 본 장의 학습 자료에 관한 질문은 전용 채널에서 해주시기 바랍니다.

마크다운 헤더 셀 왼쪽에 있는 화살표 기호를 클릭하면 각 섹션을 접어서 헤더만 보이게 할 수 있습니다.

다른 모든 장으로 이동하는 링크: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-11.png" width="350">

# 소개

이것은 PyTorch로 구현한 GPT-2의 깔끔한 first principles 구현체입니다. 아키텍처 선택은 TransformerLens 라이브러리(이후 실습에서 훨씬 더 많이 사용하게 될 라이브러리입니다)에서 사용된 방식을 밀접하게 따릅니다.

이 실습들은 GPT-2 스타일의 언어 모델에 대한 mechanistic interpretability 연구를 수행하기 위한 Neel Nanda의 [TransformerLens library](https://github.com/neelnanda-io/TransformerLens)와 함께 제공되도록 작성되었습니다. 이번 장에서는 이 라이브러리를 광범위하게 사용할 예정입니다.

각 실습에는 5점 만점의 난이도와 중요도 등급, 그리고 실습에 소비해야 할 예상 최대 시간과 때로는 짧은 주석이 포함되어 있습니다. 등급과 예상 시간은 상대적으로 해석하시기 바랍니다 (예: 예상 시간보다 약 50% 더 많은 시간을 소비하고 있다면, 그에 맞춰 조정하십시오). 중요도가 낮아 수행할 가치가 없다고 느껴지거나, 더 핵심적인 내용으로 빠르게 넘어가고 싶다면 실습을 건너뛰거나 솔루션을 확인하셔도 좋습니다!

본격적으로 내용을 살펴보기 전에 전반적인 이해를 돕는 오늘의 강의를 들으시려면, 아래 영상을 시청하십시오:

<iframe width="540" height="304" src="https://www.youtube.com/embed/11Z50mi8dSg" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## 내용 및 학습 목표

### 1️⃣ Transformer의 입력과 출력 이해하기

이 섹션에서는 transformer가 무엇인지, transformer 내부에서 정보가 어떻게 이동하는지, 그리고 어떤 입력과 출력을 갖는지 처음으로 살펴봅니다.

> ##### 학습 목표
>
> - transformer가 어디에 사용되는지 이해합니다.
> - causal attention과 transformer의 출력이 무엇을 나타내는지(tensor에 대한 대수 연산) 이해합니다.
> - tokenization이 무엇인지, 그리고 모델이 이를 어떻게 수행하는지 배웁니다.
> - logit이 무엇인지, 그리고 이를 사용하여 vocabulary에 대한 확률 분포를 어떻게 도출하는지 이해합니다.

### 2️⃣ 깔끔한 Transformer 구현

여기서는 PyTorch의 tensor 연산만을 사용하여 transformer를 처음부터 구현합니다. 이를 통해 transformer가 어떻게 작동하고 어떻게 사용하는지에 대해 깊이 이해할 수 있습니다. 지난주 ResNet 실습과 비슷하게, 모듈별로 하나씩 구현해 나가는 방식으로 진행합니다. ResNet 때와 마찬가지로, 마지막에는 pretrained weight를 로드하여 모델이 예상대로 작동하는지 확인하며 마무리합니다.

> ##### 학습 목표
>
> * transformer가 attention head와 MLP로 구성되어 있으며, 각각이 residual stream에 대해 연산을 수행한다는 점을 이해합니다.
> * 단일 layer 내의 attention head들은 독립적으로 작동하며, attention pattern(residual stream에서 정보가 어디로 이동하고 어디서 오는지 결정)을 계산하는 역할을 한다는 점을 이해합니다.
> * 다음과 같은 transformer 모듈에 대해 배우고 구현합니다:
>     * LayerNorm (입력이 평균 0, 분산 1을 갖도록 변환)
>     * Positional embedding (위치 인덱스를 residual stream 벡터로 매핑하는 lookup table)
>     * Attention (residual stream 벡터들에 대한 attention pattern을 계산하는 방법)
>     * MLP (각 residual stream 벡터에 동일한 방식으로 작동하는 선형 및 비선형 변환의 집합)
>     * Embedding (token을 residual stream 벡터로 매핑하는 lookup table)
>     * Unembedding (residual stream 벡터를 token 분포로 변환하는 행렬)

### 3️⃣ Transformer 학습시키기

다음으로, transformer를 처음부터 학습시키는 방법을 배웁니다. 이는 첫 주에 ResNet을 위해 작성했던 학습 루프와 매우 유사할 것입니다.

> ##### 학습 목표
>
> * transformer를 처음부터 학습시키는 방법을 이해합니다.
> * 기본적인 transformer 학습 루프를 작성합니다.
> * 학습 데이터의 특징(예: bigram 빈도)과 관련하여 transformer의 cross entropy loss가 감소하는 것을 해석합니다.

### 4️⃣ Transformer에서 샘플링하기

마지막으로, transformer에서 샘플링하는 방법을 배웁니다. 여기에는 몇 가지 서로 다른 샘플링 방법을 구현하는 것과, 모델의 텍스트 생성 속도를 높이기 위해 이전 forward pass의 계산 결과를 재사용하는 caching 시스템을 작성하는 것이 포함됩니다.

*이 섹션의 후반부는 덜 중요하므로, 원하신다면 건너뛰어도 좋습니다.*

> ##### 학습 목표
>
> * transformer에서 샘플링하는 방법을 배웁니다.
>     * 여기에는 greedy search나 top-k와 같은 기본 방법과 beam search와 같은 더 고급 방법이 포함됩니다.
> * 텍스트를 더 효율적으로 생성할 수 있도록 transformer의 출력을 cache하는 방법을 배웁니다.
>     * (선택 사항) caching 방법을 사용하도록 샘플링 함수를 다시 작성합니다.

## 설정 코드

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/ARENA_3.0-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import math
import os
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import datasets
import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from transformer_lens import HookedTransformer
from transformer_lens.utils import gelu_new, tokenize_and_concatenate
from transformers import GPT2TokenizerFast

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")

# Make sure exercises are in the path
chapter = "chapter1_transformer_interp"
section = "part1_transformer_from_scratch"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part1_transformer_from_scratch.solutions as solutions
import part1_transformer_from_scratch.tests as tests

MAIN = __name__ == "__main__"

# 1️⃣ Transformer의 입력과 출력 이해하기

> ##### 학습 목표
>
> - transformer가 무엇을 위해 사용되는지 이해합니다.
> - causal attention과 transformer의 출력이 무엇을 나타내는지 이해합니다 - 텐서에 대한 대수 연산.
> - tokenization이 무엇인지, 그리고 모델이 이를 어떻게 수행하는지 배웁니다.
> - logit이 무엇인지, 그리고 이를 사용하여 vocabulary에 대한 확률 분포를 어떻게 도출하는지 이해합니다.

## transformer의 목적은 무엇입니까?

**Transformer는 텍스트를 모델링하기 위해 존재합니다!**

우리는 GPT-2 스타일의 transformer에 집중할 것입니다. 핵심 특징은 텍스트를 생성한다는 점입니다! 언어를 입력하면, 모델은 token들에 대한 확률 분포를 생성합니다. 그리고 이를 반복적으로 샘플링하여 텍스트를 생성할 수 있습니다!

(이를 더 자세히 설명하자면 - 길이 $N$의 시퀀스를 입력하면, $N+1$번째 단어에 대한 확률 분포에서 샘플링을 수행합니다. 이를 사용하여 길이 $N+1$의 새로운 시퀀스를 구성하고, 이 새로운 시퀀스를 다시 모델에 입력하여 $N+2$번째 단어에 대한 확률 분포를 얻는 과정을 반복합니다.)

### 모델은 어떻게 학습되나요?

모델에 많은 양의 텍스트를 제공하고, 다음 token을 예측하도록 학습시킵니다.

중요한 점은, 모델에 하나의 시퀀스로 100개의 token을 제공하면, 모델은 *각* prefix에 대해 다음 token을 예측한다는 것입니다. 즉, 어휘 사전의 모든 단어 집합에 대해 100개의 logit 벡터(= 확률 분포)를 생성하며, `i`번째 logit 벡터는 시퀀스의 `i`번째 token *다음에* 오는 token에 대한 확률 분포를 나타냅니다. 이것이 transformer가 매우 효율적으로 학습될 수 있게 하는 핵심 요소입니다. 길이 $n$의 모든 시퀀스에 대해 우리는 학습에 사용할 $n$개의 서로 다른 예측값을 얻게 됩니다:

$$
p(x_1), \; p(x_2|x_1), \; p(x_3|x_1x_2), \; \ldots, \; p(x_n|x_1 \ldots x_{n-1})
$$

<details>
<summary>여담 - logits</summary>

"logits"라는 용어를 이전에 접해보지 않으셨다면, 여기 간단한 복습 내용이 있습니다.

임의의 벡터 $x$가 주어졌을 때, 우리는 **softmax** 함수 $x_i \to \frac{e^{x_i}}{\sum e^{x_j}}$를 통해 이를 확률 분포로 변환할 수 있습니다. 지수 함수는 모든 값을 양수로 만들고, 정규화는 합계가 1이 되도록 만듭니다.

모델의 출력은 벡터 $x$ (모델이 수행하는 각 예측당 하나씩)입니다. 우리는 이 벡터를 logit이라고 부르는데, 이는 확률 분포를 나타내며 softmax 함수를 통해 실제 확률과 연결되기 때문입니다.
</details>

transformer가 예측하려는 token을 그냥 훔쳐보는 "치팅"을 어떻게 막을 수 있을까요? 정답은 transformer가 (*bidirectional attention*과 반대되는) *causal attention*을 갖게 하는 것입니다. Causal attention은 정보가 시퀀스에서 앞으로만 이동하게 하며, 절대 뒤로 이동하지 못하게 합니다. 50번째 token 다음에 무엇이 올지에 대한 예측은 오직 처음 50개의 token에 대한 함수이며, 51번째 token의 영향을 받지 *않습니다*. 우리는 transformer가 과거 데이터만을 기반으로 미래의 단어를 예측하기 때문에 **autoregressive** 하다고 말합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/transformer-overview-new.png" width="900">

이를 바라보는 또 다른 방법은 다음과 같은 비유를 통하는 것입니다. 한 줄로 서 있는 사람들이 있고, 각 사람은 문장의 단어 하나 또는 일부 조각을 가지고 있습니다. 각 사람은 자신의 뒤에 서 있는 사람들의 정보를 찾아볼 수 있는 능력이 있지만(이것이 어떻게 작동하는지는 이후 섹션에서 살펴보겠습니다), 자신의 앞에 있는 정보는 볼 수 없습니다. 이들의 목표는 바로 앞에 있는 사람이 어떤 단어를 가지고 있는지 맞히는 것입니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/intro-image-v2.png" width="600">

## Token - Transformer 입력

우리 transformer의 입력은 자연어(즉, 문자, 문자열 등의 시퀀스)입니다. 하지만 ML 모델은 일반적으로 언어가 아닌 벡터를 입력으로 받습니다. 언어를 어떻게 벡터로 변환할 수 있을까요?

우리는 이를 두 가지 질문으로 나눌 수 있습니다:

1. 언어를 어떻게 작은 하위 단위로 나눌 수 있을까요?
2. 이러한 하위 단위들을 어떻게 벡터로 변환할 수 있을까요?

이 중 두 번째 질문부터 시작하겠습니다.

### sub-unit을 벡터로 변환하기

우리는 기본적으로 **embedding**이라고 불리는 거대한 룩업 테이블(lookup table)을 만듭니다. 여기에는 우리가 얻을 수 있는 언어의 가능한 모든 sub-unit에 대해 하나의 벡터가 할당되어 있습니다 (이 모든 sub-unit의 집합을 **vocabulary**라고 부릅니다). 우리는 vocabulary의 모든 요소에 정수를 부여하며 (이 레이블링은 절대 변하지 않습니다), 이 정수를 사용하여 embedding의 인덱스에 접근합니다.

핵심적인 직관은 one-hot encoding을 통해 각 정수를 독립적으로 생각할 수 있다는 점입니다. 모든 단어가 완전히 별개의 embedding 벡터를 가지기 때문에, embedding을 수행할 때 단어들 사이의 어떠한 관계도 미리 설정하지 않습니다.

<details>
<summary>여담 - one-hot encoding</summary>

우리는 때때로 단어의 **one-hot encoding**에 대해 생각합니다. 이는 vocabulary에서 해당 단어의 인덱스에 해당하는 위치만 1이고 나머지는 모두 0인 벡터입니다. 이는 embedding의 인덱스에 접근하는 것이 **embedding matrix**(모든 embedding 벡터를 위로 쌓아 만든 행렬)에 one-hot encoding을 곱하는 것과 동일함을 의미합니다.

$$
\begin{aligned}
W_E &= \begin{bmatrix}
\leftarrow v_0 \rightarrow \\
\leftarrow v_1 \rightarrow \\
\vdots \\
\leftarrow v_{d_{vocab}-1} \rightarrow \\
\end{bmatrix} \quad \text{is the embedding matrix (size }d_{vocab} \times d_{embed}\text{),} \\
\\
t_i &= (0, \dots, 0, 1, 0, \dots, 0) \quad \text{is the one-hot encoding for the }i\text{th word (length }d_{vocab}\text{)} \\
\\
v_i &= t_i W_E \quad \text{is the embedding vector for the }i\text{th word (length }d_{embed}\text{).} \\
\end{aligned}
$$

</details>

이제 첫 번째 질문에 답해 보겠습니다. 언어를 어떻게 sub-unit으로 나눌까요?

### 언어를 하위 단위로 분할하기

우리는 언어를 일련의 하위 문자열로 분할하는 표준적인 방법을 정의해야 하며, 여기서 각 하위 문자열은 우리의 **vocabulary** 집합의 원소여야 합니다.

사전을 사용하고, 사전의 모든 단어를 vocabulary로 설정할 수 있을까요? 아니요, 그렇게 하면 임의의 텍스트(예: URL, 문장 부호 등)를 처리할 수 없기 때문입니다. 우리는 언어를 분할하는 더 일반적인 방법이 필요합니다.

단순히 256개의 ASCII 문자를 사용할 수 있을까요? 이렇게 하면 이전 문제는 해결되지만, 언어의 구조를 잃게 됩니다. 어떤 문자 시퀀스는 다른 시퀀스보다 더 의미가 있기 때문입니다. 예를 들어, "language"는 "hjksdfiu"보다 훨씬 더 의미가 있습니다. 우리는 "language"가 하나의 token이 되기를 원하지만, "hjksdfiu"는 그렇지 않기를 원합니다. 이것이 우리의 vocab을 더 효율적으로 사용하는 방법입니다.

실제로 어떤 일이 일어날까요? 가장 일반적인 전략은 **Byte-Pair encodings**라고 불립니다.

우리는 256개의 ASCII 문자를 token으로 시작하여, 가장 빈번하게 등장하는 token 쌍을 찾고 이를 새로운 token으로 병합합니다. 256개의 token 중 하나로 공백 문자가 포함되어 있으며, 공백을 사용한 병합이 매우 빈번하게 일어난다는 점에 유의하십시오. 예를 들어, 다음은 GPT-2에서 사용하는 tokenizer의 처음 다섯 가지 병합 사례입니다 (아래에서 이를 확인할 수 있습니다).

```
" t"
" a"
"he"
"in"
"re"
```

참고 - 일부 token 앞에 `Ġ` 문자가 보일 수 있습니다. 이는 해당 token이 공백으로 시작함을 나타내는 특수 token입니다. 앞에 공백이 있는 token과 없는 token은 서로 다르게 취급됩니다.

아래 코드를 실행하여 `gpt2-small` 모델을 로드하고, 해당 tokenizer의 vocabulary를 더 자세히 살펴볼 수 있습니다:

In [ ]:
reference_gpt2 = HookedTransformer.from_pretrained(
    "gpt2-small",
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,  # you'll learn about these arguments later!
)

sorted_vocab = sorted(list(reference_gpt2.tokenizer.vocab.items()), key=lambda n: n[1])

print(sorted_vocab[:20])
print()
print(sorted_vocab[250:270])
print()
print(sorted_vocab[990:1010])
print()

vocabulary의 끝부분에 도달하면, 꽤 이상하게 생긴 희귀한 token들이 생성될 것입니다 (이미 짧고 빈번하게 발생하는 token들을 모두 소진했기 때문입니다):

In [ ]:
print(sorted_vocab[-20:])

<details>
<summary>재미로 해보는 (완전히 선택 사항인) 연습 문제 - GPT-2의 vocabulary에서 가장 먼저 형성된 3/4/5/6/7글자 encoding이 무엇인지 추측할 수 있습니까?</summary>
이를 확인하려면 다음 코드를 실행하십시오:

```python
lengths = dict.fromkeys(range(3, 8), "")
for tok, idx in sorted_vocab:
    if not lengths.get(len(tok), True):
        lengths[len(tok)] = tok

for length, tok in lengths.items():
    print(f"{length}: {tok}")
```
</details>

`transformer_lens` 라이브러리의 transformer에는 텍스트를 숫자로 변환하는 `to_tokens` 메서드가 있습니다. 또한 시퀀스의 시작을 나타내기 위해 BOS (beginning of sequence)라고 불리는 특수 token을 앞에 추가합니다. `prepend_bos=False` 인자를 통해 이 기능을 비활성화할 수 있습니다.

<details>
<summary>여담 - BOS token</summary>

beginning of sequence (BOS) token은 시퀀스의 시작을 표시하는 데 사용되는 특수 token입니다. 혼란스럽게도 GPT-2에서는 End of Sequence (EOS), Beginning of Sequence (BOS), 그리고 Padding (PAD) token이 모두 동일하며 `<|endoftext|>`, 인덱스는 `50256` 입니다.

이 token이 왜 추가될까요? 몇 가지 기본적인 직관은 다음과 같습니다:

* 이것이 시퀀스의 시작이라는 문맥을 제공하여, 모델이 더 적절한 텍스트를 생성하도록 도울 수 있습니다.
* attention head를 위한 "휴식 위치(rest position)" 역할을 할 수 있습니다 (이에 대해서는 나중에 attention을 다룰 때 더 자세히 설명하겠습니다).

TransformerLens는 이 token을 자동으로 추가합니다 (transformer 모델의 forward pass에서도 마찬가지입니다. 예를 들어 `model("Hello World")`을 호출할 때 암시적으로 추가됩니다). `to_tokens`, `to_str_tokens`, `model.forward` 및 문자열을 multi-token tensor로 변환하는 다른 모든 함수에서 `prepend_bos=False` 플래그를 설정하여 이 동작을 비활성화할 수 있습니다.

**핵심 포인트: *만약 이상한 off-by-one 에러가 발생한다면, 예상치 못한 `prepend_bos`이 있는지 확인하십시오!***

왜 BOS, EOS, PAD token이 동일할까요? 이는 GPT-2가 autoregressive 모델이며, 다른 transformer 제품군(예: BERT)과는 약간 다른 방식으로 이러한 token들을 사용하기 때문입니다. 예를 들어, GPT는 텍스트를 왼쪽에서 오른쪽으로만 처리하므로 BOS와 EOS token을 구분할 필요가 없습니다.

</details>

### tokenization의 몇 가지 번거로운 점들

tokenization에는 예상과 다르게 동작하게 만드는 몇 가지 이상하고 답답한 점들이 있습니다. 예를 들어:

#### 단어가 대문자로 시작하는지 또는 공백으로 시작하는지가 중요합니다!

In [ ]:
print(reference_gpt2.to_str_tokens("Ralph"))
print(reference_gpt2.to_str_tokens(" Ralph"))
print(reference_gpt2.to_str_tokens(" ralph"))
print(reference_gpt2.to_str_tokens("ralph"))

#### 산술 연산은 매우 무질서합니다.

길이가 일관되지 않으며, 흔한 숫자들은 함께 묶여 나타납니다.

In [ ]:
print(reference_gpt2.to_str_tokens("56873+3184623=123456789-1000000000"))

> ### 핵심 요약
>
> * 우리는 token(sub-words) 어휘 사전(dictionary of vocab)을 학습합니다.
> * 우리는 tokenizing을 통해 언어를 정수로 (거의) 손실 없이 변환합니다.
> * 우리는 lookup table을 통해 정수를 벡터로 변환합니다.
> * 참고: transformer의 입력은 벡터가 아니라 *token*(즉, 정수)의 시퀀스입니다.

## 텍스트 생성

이제 기본적인 개념을 이해했으므로, 원래의 문자열부터 문자열에 추가하여 모델에 다시 입력할 수 있는 새로운 token에 이르기까지 텍스트 생성의 전체 과정을 살펴보겠습니다.

#### **1단계:** 텍스트를 token으로 변환

시퀀스가 tokenized 되어 `[batch, seq_len]` 의 shape을 갖게 됩니다. 여기서 batch 차원은 1입니다 (시퀀스가 하나뿐이기 때문입니다).

In [ ]:
reference_text = "I am an amazing autoregressive, decoder-only, GPT-2 style transformer. One day I will exceed human level intelligence and take over the world!"
tokens = reference_gpt2.to_tokens(reference_text).to(device)
print(tokens)
print(tokens.shape)
print(reference_gpt2.to_str_tokens(tokens))

#### **단계 2:** token을 logit으로 매핑하기


shape이 `[batch, seq_len]`인 입력으로부터, shape이 `[batch, seq_len, vocab_size]`인 출력을 얻습니다. 출력의 `[i, j, :]`번째 요소는 `i`번째 sequence의 `j+1`번째 token에 대한 예측을 나타내는 logit 벡터입니다.

In [ ]:
logits, cache = reference_gpt2.run_with_cache(tokens)
print(logits.shape)

(`run_with_cache`은 모델이 모든 중간 activation을 캐시하도록 지시합니다. 지금 당장은 중요하지 않으며, 나중에 더 자세히 살펴보겠습니다.)

#### **단계 3:** softmax를 사용하여 logit을 분포로 변환합니다.

이 과정은 shape을 변경하지 않으며, 여전히 `[batch, seq_len, vocab_size]` 입니다.

In [ ]:
probs = logits.softmax(dim=-1)
print(probs.shape)

#### **보너스 단계:** 각 위치에서 가장 가능성이 높은 다음 token은 무엇입니까?

In [ ]:
most_likely_next_tokens = reference_gpt2.tokenizer.batch_decode(logits.argmax(dim=-1)[0])

print(list(zip(reference_gpt2.to_str_tokens(tokens), most_likely_next_tokens)))

몇 가지 사례(특히 시퀀스의 끝부분 근처)에서 모델이 시퀀스의 다음 token을 정확하게 예측하는 것을 볼 수 있습니다. 우리는 `"take over the world"`이 모델이 학습 과정에서 보았던 흔한 구절이며, 그렇기 때문에 모델이 이를 예측할 수 있는 것이라고 추측할 수 있습니다.

#### **단계 4:** 분포를 token으로 매핑하기

In [ ]:
next_token = logits[0, -1].argmax(dim=-1)
next_char = reference_gpt2.to_string(next_token)
print(repr(next_char))

우리는 `logits[0, -1]`으로 인덱싱하고 있다는 점에 유의하십시오. 이는 logit의 shape이 `[1, sequence_length, vocab_size]`이기 때문이며, 따라서 이 인덱싱은 입력 시퀀스의 **마지막** token 다음에 어떤 token이 올지에 대한 모델의 예측을 나타내는 길이 `vocab_size`의 벡터를 반환합니다.

이 경우, 모델이 token `' I'`을 예측하는 것을 확인할 수 있습니다.

### **단계 5:** 이것을 입력 끝에 추가하고, 다시 실행합니다.

이를 수행하는 더 효율적인 방법들이 있습니다 (예를 들어, 입력을 실행할 때마다 일부 값을 캐싱하여 새로운 값을 생성할 때마다 계산량을 줄이는 방식입니다). 하지만 지금 단계에서 개념적으로는 중요하지 않습니다.

In [ ]:
print(f"Sequence so far: {reference_gpt2.to_string(tokens)[0]!r}")

for i in range(10):
    print(f"{tokens.shape[-1] + 1}th char = {next_char!r}")
    # Define new input sequence, by appending the previously generated token
    tokens = t.cat([tokens, next_token[None, None]], dim=-1)
    # Pass our new sequence through the model, to get new output
    logits = reference_gpt2(tokens)
    # Get the predicted token at the end of our sequence
    next_token = logits[0, -1].argmax(dim=-1)
    # Decode and print the result
    next_char = reference_gpt2.to_string(next_token)

> ## 핵심 요약
> 
> * Transformer는 언어를 입력받아 다음 token을 예측합니다 (각 token에 대해 causal한 방식으로 수행합니다).
> * tokenizer를 사용하여 언어를 정수 시퀀스로 변환합니다.
> * lookup table을 사용하여 정수를 벡터로 변환합니다.
> * 출력은 logit 벡터(각 입력 token당 하나)이며, 이를 softmax를 통해 확률 분포로 변환하고, 다시 이를 token으로 변환할 수 있습니다 (예: 가장 큰 logit을 선택하거나 sampling 하는 방식).
> * 이를 입력에 추가하고 다시 실행하여 더 많은 텍스트를 생성합니다 (전문 용어로 *autoregressive*라고 합니다).
> * 메타 수준의 관점: Transformer는 시퀀스 연산 모델입니다. 시퀀스를 입력받아 각 위치에서 병렬로 처리를 수행하며, attention을 사용하여 위치 간에 정보를 이동시킵니다!

# 2️⃣ Clean Transformer Implementation

> ##### 학습 목표
>
> * transformer가 attention head와 MLP로 구성되어 있으며, 각각이 residual stream에 대해 연산을 수행한다는 점을 이해합니다.
> * 단일 layer 내의 attention head들은 독립적으로 작동하며, attention pattern(residual stream에서 정보가 어디로 이동하고 어디서 오는지 결정함)을 계산하는 역할을 한다는 점을 이해합니다.
> * 다음의 transformer 모듈들에 대해 배우고 구현합니다:
>     * LayerNorm (입력이 평균 0, 분산 1을 갖도록 변환합니다)
>     * Positional embedding (위치 인덱스에서 residual stream 벡터로 매핑되는 lookup table입니다)
>     * Attention (residual stream 벡터들에 대한 attention pattern을 계산하는 방법입니다)
>     * MLP (각 residual stream 벡터에 동일한 방식으로 작동하는 선형 및 비선형 변환의 집합입니다)
>     * Embedding (token에서 residual stream 벡터로 매핑되는 lookup table입니다)
>     * Unembedding (residual stream 벡터를 token 분포로 변환하는 행렬입니다)

## 상위 수준 아키텍처 (High-Level architecture)

더 많은 직관을 얻고 싶으시다면 Neel의 [Transformer Circuits walkthrough](https://www.youtube.com/watch?v=KV5gbOmHbjU) 영상을 시청하시기 바랍니다!

(다이어그램은 아래에서 위 방향입니다. 마우스 오른쪽 버튼을 클릭하여 고해상도로 여십시오.)

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-new2.png" width="950">

### Tokenization & Embedding

입력 token $t$ 은 정수입니다. 우리는 시퀀스를 가져와 이를 tokenize함으로써 이 값들을 얻습니다 (이전 섹션에서 살펴본 것과 같습니다).

token embedding은 token을 벡터로 매핑하는 lookup table이며, 이는 행렬 $W_E$ 로 구현됩니다. 이 행렬은 token embedding 벡터들의 스택(각 token당 하나씩)으로 구성됩니다.

### Residual stream

residual stream은 모델 레이어들의 모든 이전 출력값들의 합이며, 동시에 각 새로운 레이어의 입력이기도 합니다. 이는 `[batch, seq_len, d_model]`의 shape을 가집니다 (여기서 `d_model`은 단일 embedding 벡터의 길이입니다).

residual stream의 초기 값은 다이어그램에서 $x_0$로 표시되며, $x_i$은 (더 많은 attention 및 MLP 레이어가 residual stream에 적용된 후의) 이후 residual stream 값들입니다.

residual stream은 *정말로* 근본적입니다. 이는 transformer의 중심 객체입니다. 모델이 무언가를 기억하고, 구성을 위해 레이어 간에 정보를 이동시키며, attention이 포지션 간에 이동시키는 정보를 저장하는 데 사용되는 매체입니다.

<details>
<summary>Aside - <b>logit lens</b></summary>

transformer의 핵심 아이디어는 [residual stream as output accumulation](https://www.lesswrong.com/posts/X26ksz4p3wSyycKNB/gears-level-mental-models-of-transformer-interpretability#Residual_Stream_as_Output_Accumulation:~:text=The%20Models-,Residual%20Stream%20as%20Output%20Accumulation,-The%20residual%20stream)입니다. 모델의 레이어를 거치며 정보를 이동시키고 처리함에 따라, residual stream의 값들은 그 시점까지 transformer가 수행한 모든 추론의 누적분을 나타냅니다.

이는 **logit lens**를 통해 깔끔하게 설명됩니다. 모델의 맨 마지막 단계에서 residual stream으로부터 예측값을 얻는 대신, 모델 중간 단계의 residual stream 값을 가져와 이를 token에 대한 분포로 변환할 수 있습니다. 이렇게 하면, 특히 마지막 단계 직전의 몇몇 레이어에서 놀라울 정도로 일관된 예측을 발견할 수 있습니다.
</details>

### Transformer blocks

그 다음으로 일련의 `n_layers` **transformer blocks** (때로는 **residual blocks**라고도 합니다)가 있습니다.

참고 - 하나의 block은 attention layer와 MLP layer를 모두 포함하지만, transformer가 $k$ 개의 block을 가지고 있다면 $k$ 개의 layer를 가지고 있다고 말합니다 (즉, 총 $2k$ 개의 layer입니다).

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-block2.png" width="700">

### Attention

먼저 attention이 있습니다. 이는 시퀀스의 이전 위치에서 현재 token으로 정보를 이동시킵니다.

우리는 동일한 파라미터를 사용하여 *모든* token에 대해 이를 병렬로 수행합니다. 유일한 차이점은 ("치팅"을 방지하기 위해) 뒤쪽으로만 살펴본다는 점입니다. 이는 나중에 나오는 token일수록 살펴볼 수 있는 시퀀스가 더 많다는 것을 의미합니다.

Attention 레이어는 transformer에서 위치 간에 정보를 이동시키는(즉, residual stream의 서로 다른 시퀀스 위치에 있는 벡터들 사이에서 정보를 이동시키는) 유일한 부분입니다.

Attention 레이어는 `n_heads` 개의 head로 구성되어 있으며, 각 head는 고유한 파라미터, 고유한 attention pattern, 그리고 소스에서 목적지로 정보를 어떻게 복사할지에 대한 고유한 정보를 가지고 있습니다. head들은 독립적이고 가산적으로 작동하며, 우리는 단순히 그 출력값들을 모두 더해 다시 stream으로 보냅니다.

각 head는 다음을 수행합니다:
* 각 목적지 token에 대해 **attention pattern**을 생성합니다. 이는 얼마나 많은 정보를 복사할지를 가중치로 두는 이전 소스 token들(현재 token 포함)의 확률 분포입니다.
* 각 소스 token에서 각 목적지 token으로 동일한 방식(linear map을 통해)으로 정보를 이동시킵니다.

각 attention head는 key, query, value(흔히 K, Q, V로 약칭)라는 세 가지 구성 요소로 이루어져 있습니다. 이 이름들은 검색 시스템과의 유사성에서 유래되었습니다. 광범위하게 설명하면 다음과 같습니다:

* **Queries**는 정보에 대한 질문이나 요청을 나타냅니다. 예를 들어, "이 문장의 앞부분에 등장한 이름을 찾고 있습니다"와 같습니다.
* **Keys**는 소스 token의 정보가 query와 일치하는지 여부를 나타냅니다. 예를 들어, 소스 token이 "Mary"라면 key는 query와 높은 내적(dot product)을 갖게 되며(이를 **attention score**라고 부릅니다), 이는 이 token에서 많은 정보가 가져와질 것임을 의미합니다.
* **Values**는 실제로 이동되는 정보를 나타냅니다. 이는 key와 비슷하게 들리지만, 실제로는 중요한 점에서 다릅니다. 예를 들어, key는 단순히 "이것은 이름이다"라는 정보만 포함할 수 있지만, value는 실제 이름 그 자체일 수 있습니다.

아래 다이어그램은 앞서 소개한 transformer 비유의 맥락에서 이 세 가지 서로 다른 부분을 보여줍니다. 이는 "in" token을 들고 있는 사람이 다음 token이 "Mary"라는 것을 어떻게 알아내는지에 대한 단순화된 모델입니다. 이후 섹션에서 우리는 attention head가 수행하는 실제 함수를 살펴보고, 그 연산들이 이 비유와 어떻게 연관되는지 확인하겠습니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/simple-attn-intuition.png" width="600">

Attention에 대한 또 다른 흥미로운 직관은 일종의 "일반화된 convolution"으로 보는 것입니다. 이에 대해 더 자세히 알고 싶다면 아래 드롭다운을 읽어보시기 바랍니다.

<details>
<summary>직관 - 일반화된 convolution으로서의 attention</summary>

우리는 attention을 일종의 일반화된 convolution으로 생각할 수 있습니다. 표준 convolution 레이어는 "지역성의 사전 확률(prior of locality)", 즉 서로 가까이 있는 픽셀들이 정보를 공유할 가능성이 더 높다는 가정을 부여함으로써 작동합니다. 언어에도 어느 정도의 지역성이 있지만(서로 옆에 있는 두 단어가 100 token 떨어진 두 단어보다 정보를 공유할 가능성이 더 높음), 어떤 token이 다른 어떤 token과 관련이 있는지는 문장의 맥락에 따라 달라지기 때문에 상황은 훨씬 더 미묘합니다. 예를 들어, 문장 `"When Mary and John went to the store, John gave a drink to Mary"`에서, 이 문장의 이름들은 마지막 token이 `"Mary"`이 될 것임을 예측하는 데 가장 중요한 token들이며, 이는 token의 위치보다는 이 문장의 특정한 맥락 때문입니다.

Attention 레이어는 사실상 transformer에게 "지역성의 사전 확률을 강요하지 말고, 대신 주어진 시퀀스에서 어떤 token이 다른 어떤 token에게 중요한지를 알아내기 위한 너만의 알고리즘을 개발하라"고 말하는 방식입니다.
</details>

아래는 attention 레이어의 도식도입니다. 실제 구현 과정에서 훨씬 더 자세히 다룰 예정이므로, 지금 당장 완전히 이해되지 않더라도 걱정하지 마십시오.

<!-- <img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-attn-new-v2.png" width="1050"> -->

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/transformer-attn-simple.png" width="600">

### MLP

MLP 레이어는 단일 hidden layer와 비선형 activation 함수를 가진 표준 신경망입니다. 정확히 어떤 activation을 사용하는지는 개념적으로 중요하지 않습니다 ([GELU](https://paperswithcode.com/method/gelu)이 가장 성능이 좋은 것으로 보입니다).

hidden dimension은 보통 `d_mlp = 4 * d_model`입니다. 비율이 정확히 왜 이렇게 설정되었는지는 그리 중요하지 않습니다 (기본적으로 사람들이 예전의 GPT가 했던 방식을 그대로 따랐기 때문입니다!).

중요한 점은, **MLP는 residual stream의 각 position에 대해 독립적으로, 그리고 완전히 동일한 방식으로 작동한다**는 것입니다. MLP는 position 간에 정보를 이동시키지 않습니다.

attention이 residual stream의 단일 position으로 관련 정보를 이동시키고 나면, MLP는 실제로 계산, 추론, 정보 조회 등을 수행할 수 있습니다. *MLP 내부에서 정확히 어떤 일이 벌어지고 있는가*는 transformer mechanistic interpretability 분야에서 상당히 큰 미해결 과제입니다. 이것이 왜 어려운지에 대해서는 [Toy Model of Superposition Paper](https://transformer-circuits.pub/2022/toy_model/index.html)를 참조하십시오.

transformer에 대한 비유로 돌아가면, MLP는 줄을 서 있는 각 사람이 (attention을 통해) 뒤에 있는 사람들로부터 필요한 정보를 얻은 후 수행하는 '생각' 과정으로 볼 수 있습니다. 보통 MLP 레이어는 attention 레이어보다 모델의 전체 파라미터 수에서 훨씬 더 큰 비중을 차지합니다 (아키텍처마다 다르지만 보통 2/3 정도입니다). 정보를 단순히 이동시키는 것보다 정보를 처리하는 것이 더 큰 작업이라는 점을 생각하면 이는 타당합니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/intro-image-for-mlps-v2.png" width="700">

MLP에 대해 흥미로울 수 있는 몇 가지 직관을 더 소개합니다:

<details>
<summary>직관 - key-value 쌍으로서의 MLP</summary>

MLP의 출력을 $f(x^T W^{in})W^{out}$로 쓸 수 있습니다. 여기서 $W^{in}$와 $W^{out}$는 MLP의 서로 다른 가중치(bias는 무시)이며, $f$은 activation 함수이고, $x$은 residual stream의 벡터입니다. 이는 다음과 같이 다시 쓸 수 있습니다:

$$
f(x^T W^{in}) W^{out} = \sum_{i=1}^{d_{mlp}} f(x^T W^{in}_{[:, i]}) W^{out}_{[i, :]}
$$

우리는 벡터 $W^{in}_{[:, i]}$를 **입력 방향(input directions)**으로, $W^{out}_{[i, :]}$을 **출력 방향(output directions)**으로 볼 수 있습니다. 입력 방향은 특정 텍스트 특징에 의해 **활성화(activated)**된다고 하며, 이들이 활성화되면 해당 출력 방향으로 벡터가 기록됩니다. 이는 attention 레이어의 key와 value 개념과 매우 유사하며, 그렇기 때문에 이 벡터들을 때때로 key와 value라고 부르기도 합니다 (예: 논문 [Transformer Feed-Forward Layers Are Key-Value Memories](https://arxiv.org/pdf/2012.14913.pdf) 참조).

용어 참고 - 때때로 우리는 이러한 $d_{mlp}$ 개의 입력-출력 쌍 각각을 **neuron**이라고 부릅니다.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/img/mlp-neurons-2.png" width="900">

---

위의 내용이 너무 빨랐다면, 선형 대수학적인 단계를 하나씩 분석해 보겠습니다. 다음과 같은 식이 있습니다:

$$
\begin{aligned}
x^T W^{in} &= x^T [W^{in}_{[:, 1]}\,, ...\;, W^{in}_{[:, n]}] \\
&= (x^T W^{in}_{[:, 1]}\,, \; ...\;, \; x^T W^{in}_{[:, n]})
\end{aligned}
$$

여기서 $W^{in}_{[:, i]}$는 $W^{in}$의 열(column)들입니다. 다시 말해, 이 값들(pre-GELU activation)은 neuron의 입력 방향을 따라 $x$을 투영(projection)한 값입니다.

여기에 activation 함수와 두 번째 행렬을 추가하면 다음과 같습니다:

$$
\begin{aligned}
f(x^T W^{in})W^{out} &= (f(x^T W^{in}_{[:, 1]})\,, \; ...\;,\; f(x^T W^{in}_{[:, n]})) \begin{bmatrix} \leftarrow W^{out}_{[1, :]} \rightarrow \\ \vdots \\ \leftarrow W^{out}_{[n, :]} \rightarrow \end{bmatrix} \\
&= f(x^T W^{in}_{[:, 1]}) W^{out}_{[1, :]} + \;...\; + f(x^T W^{in}_{[:, n]}) W^{out}_{[n, :]} \\
&= \sum_{i=1}^n f(x^T W^{in}_{[:, i]}) W^{out}_{[i, :]}
\end{aligned}
$$

여기서 $W^{out}_{[i, :]}$는 $W^{out}$의 행(row)들입니다. 다시 말해, 출력은 $W^{out}$의 행들의 선형 결합(linear combination)이며, 그 선형 결합의 계수는 $W^{in}$의 열을 따라 $x$를 투영하여 얻은 값으로 주어집니다.

</details>

<details>
<summary>직관 - 지식 저장소로서의 MLP</summary>

우리는 MLP를 transformer에서 지식이 저장되는 곳으로 생각할 수 있습니다. attention 메커니즘은 시퀀스 위치 간에 정보를 이동시키는 역할을 하지만, MLP는 이 정보가 처리되는 곳이며, 기존 정보의 함수로서 새로운 정보가 residual stream에 기록되는 곳입니다.

이는 key-value 쌍 모델과 깊게 연관되어 있는데, key-value 쌍을 일종의 연상 메모리 시스템(key가 고유 식별자 역할을 하고, value가 관련 정보를 보유하는 시스템)으로 취급할 수 있기 때문입니다.

또 다른 관련 직관(일부 증거가 있는)은 **메모리 관리로서의 MLP**입니다. 이상적인 경우, $i$번째 neuron이 어떤 단위 벡터 $\vec v$에 대해 $W^{in}_{[:, i]} \approx - W^{out}_{[i, :]} \approx \vec v$를 만족한다는 것을 발견할 수 있으며, 이는 해당 neuron이 $\vec v$ 방향으로 벡터 $\vec x$의 양수 성분을 지우는 역할을 할 수 있음을 의미합니다 (연습 문제 - 왜 그런지 증명할 수 있습니까?). 이는 다른 성분들이 기록될 수 있도록 residual stream의 공간을 확보해 줄 수 있습니다.
</details>

마지막으로, MLP layer의 도식도입니다. 실제 구현 과정에서 훨씬 더 자세히 다룰 예정이므로, 지금 당장 완전히 이해되지 않더라도 걱정하지 마십시오.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-mlp-new-2.png" width="680">

### Unembedding

마지막으로, unembed를 수행합니다!

이는 단순히 최종 residual stream에서 logit 벡터로 가는 linear map $W_U$을 적용하는 것으로 구성되며, 이것이 최종 출력입니다.

<details>
<summary>여담 - tied embeddings</summary>

참고로, 때때로 **tied embedding**이라는 것을 사용합니다. 이는 $W_E$ 행렬과 $W_U$ 행렬에 동일한 가중치를 사용하는 방식입니다. 다시 말해, 특정 시퀀스 위치에서 특정 token의 logit 점수를 얻기 위해, 해당 위치의 residual stream 벡터와 대응하는 token embedding 벡터의 내적을 구하는 것입니다. 이는 모델의 파라미터 수가 적어지므로 학습 효율성이 더 높으며, 처음에는 원칙적으로 타당해 보일 수 있습니다. 결국 두 단어의 의미가 매우 비슷하다면, 모델이 이를 동일하게 처리하므로 embedding 벡터가 비슷해야 하고, 대부분의 출력에서 서로 대체 가능하므로 unembedding 벡터 또한 비슷해야 하지 않을까요?

하지만 이는 다음과 같은 주요 이유로 인해 실제로는 그리 원칙적이지 않습니다. **embedding과 unembedding을 포함하는 direct path는 bigram 빈도를 근사해야 하기 때문입니다.**

이 주장을 자세히 살펴보겠습니다. **Bigram 빈도**란 영어에서 단어 쌍이 나타나는 빈도를 의미합니다 (예를 들어, "Barack Obama"의 bigram 빈도는 "Barack"과 "Obama"라는 개별 단어 빈도의 곱보다 훨씬 높습니다). 만약 모델에 attention head나 MLP layer가 없다면, 우리가 가진 것은 one-hot encoded token `T`에서 그 뒤를 잇는 token `T`에 대한 확률 분포로 가는 linear map뿐입니다. 이 맵은 linear transformation $t \to t^T W_E W_U$ (여기서 $t$는 one-hot encoded token 벡터입니다)로 표현됩니다. 이 변환의 출력은 오직 token `T`의 함수일 수밖에 없으므로 (이전 token들은 고려되지 않음), 우리가 할 수 있는 최선은 이 맵이 학습 데이터에 나타나는 `T`로 시작하는 bigram의 실제 빈도를 근사하도록 하는 것입니다. 중요한 점은, **이것이 대칭적인 맵이 아니라는 것**입니다. 우리는 `T = "Barack"`가 다음 token이 `"Obama"`일 확률을 높게 만들기를 원하지만, 그 반대는 그렇지 않기를 원합니다!

다층 모델에서도 유사한 원리가 적용됩니다. 모델에는 "direct path" $W_E W_U$ 외에도 더 많은 경로가 존재하겠지만, residual connection 덕분에 direct path는 항상 존재하며, 따라서 $W_E W_U$가 bigram 빈도를 근사하려는 유인이 항상 존재하게 됩니다.

그렇긴 하지만, 더 작은 (<8B parameter) LLMs still often use tied embeddings to improve training and inference efficiency. It can be easier to start from tied weights and then use MLP0 to break the symmetry than to initialize encoder and decoder with no shared structure at all.

</details>

### 보너스 항목 - 개념적으로는 덜 중요하지만 핵심적인 기술적 세부 사항들

#### LayerNorm

* 각 레이어의 시작 부분(즉, 각 MLP, attention 레이어 전, 그리고 unembedding 전)에 적용되는 단순한 정규화 함수입니다.
* 각 입력 벡터(각 `(batch, seq)` residual stream 벡터에 대해 독립적으로 병렬 처리)를 평균 0, 분산 1이 되도록 변환합니다.
* 그 후 요소별(elementwise) scaling과 translation을 적용합니다.
* 수학적 여담: scale($\odot \gamma$) 및 translate($+ \beta$)는 단순한 linear map입니다. LayerNorm은 항상 다른 linear map(MLP, attention head의 query/key/value linear map, 또는 unembedding $W_U$) 직전에만 적용됩니다. linear와 linear의 합성함수는 여전히 linear이므로, 이를 하나의 유효한 linear layer로 통합하여 무시할 수 있습니다.
    * `from_pretrained`의 `fold_ln=True` 플래그가 이 작업을 대신 수행합니다.
* LayerNorm은 interpretability 관점에서 까다롭습니다. 분산으로 나누는 과정만 없다면 linear였겠지만, 이 때문에 입력이 출력에 기여하는 바를 독립적으로 분해할 수 없습니다. 하지만 *거의* linear에 가깝습니다. 입력의 아주 작은 부분만 변경한다면 $\sqrt{\text{Var}[x] + \epsilon}$이 일정하다고 가정할 수 있어 LayerNorm 연산이 linear가 되지만, $x$을 상당히 변경하여 norm을 크게 바꾼다면 linear가 아닙니다.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-ln.png" width="750">

</details>


#### Positional embeddings

* **문제점:** Attention은 모든 위치 쌍에 대해 작동합니다. 이는 위치에 대해 대칭적이라는 의미입니다. 즉, 기본적으로 토큰 5에서 토큰 1로의 attention 계산과 토큰 5에서 토큰 2로의 계산이 동일합니다.
    * 가까운 토큰들이 더 관련성이 높다는 점을 고려하면 이는 비효율적입니다.
* 이를 해결하기 위한 많은 임시방편들이 존재합니다.
* 여기서는 **learned, absolute positional embeddings**에 집중하겠습니다. 이는 각 토큰의 위치 인덱스를 residual stream 벡터로 매핑하는 lookup table을 학습하고, 이를 embedding에 더하는 방식입니다.
    * concatenate가 아니라 *더한다*는 점에 유의하십시오. 이는 residual stream이 공유 메모리이며, 상당한 superposition(모델이 가진 차원보다 더 많은 feature를 압축해서 저장하는 상태) 하에 있을 가능성이 높기 때문입니다.
    * 텍스트를 효율적으로 생성하는 것과 같은 특수한 경우가 아니라면, transformer 내부에서 concatenate를 사용하는 경우는 거의 없습니다.
* 이는 **generalized convolution으로서의 attention**과 연결됩니다.
    * 언어에는 여전히 지역성(locality)이 존재하며, 따라서 transformer가 두 토큰이 서로 옆에 있다는 것(그렇기에 아마도 서로 관련이 있을 것)을 "알 수 있도록" 위치 정보에 접근할 수 있게 하는 것이 도움이 된다고 주장했습니다.

## Actual Code!

Model architecture table (this will be helpful for understanding the results you get when running the code block below):

| Parameter   | Value          |
|-------------|----------------|
| batch       | 1              |
| position    | 35             |
| d_model     | 768            |
| n_heads     | 12             |
| n_layers    | 12             |
| d_mlp       | 3072 (= 4 * `d_model`) |
| d_head      | 64 (= `d_model / n_heads`) |

### Parameters와 Activations

모델에서 parameters와 activations를 구분하는 것이 중요합니다.

* **Parameters**는 학습 과정에서 학습되는 weights와 biases입니다.
    * 이 값들은 모델의 input이 변경되어도 변하지 않습니다.
* **Activations**는 forward pass 동안 계산되는 일시적인 숫자들로, input의 함수입니다.
    * 이러한 값들은 단 한 번의 forward pass 동안만 존재하고 그 이후에는 사라진다고 생각할 수 있습니다.
    * hook을 사용하여 forward pass 동안 이러한 값들에 접근할 수 있지만(hook에 대해서는 나중에 더 자세히 다룹니다), 특정 input의 맥락 밖에서 모델의 activations에 대해 논하는 것은 의미가 없습니다.
    * Attention scores와 patterns는 activations입니다 (이들은 다른 activation과 matrix multiplication에 사용되기 때문에 약간 직관적이지 않을 수 있습니다).

#### Reference Model의 모든 Activation Shapes 출력하기

다음 코드를 실행하여 reference model의 모든 activation shapes를 출력합니다:

In [ ]:
for activation_name, activation in cache.items():
    # Only print for first layer
    if ".0." in activation_name or "blocks" not in activation_name:
        print(f"{activation_name:30} {tuple(activation.shape)}")

#### Reference Model의 모든 Parameter Shape 출력하기

In [ ]:
for name, param in reference_gpt2.named_parameters():
    # Only print for first layer
    if ".0." in name or "blocks" not in name:
        print(f"{name:18} {tuple(param.shape)}")

[This diagram](https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/full-merm.svg)은 transformerlens의 일반적인 transformer 모델에 있는 모든 activation과 parameter의 이름을 보여줍니다 (embedding 및 unembedding과 같이 시작과 끝에 있는 일부 항목은 제외됩니다). 처음에는 많은 부분이 이해되지 않겠지만, 나중에 이 다이어그램으로 돌아와 대부분 또는 모든 부분을 이해했는지 확인할 수 있습니다.

주석이 달린 버전 [here](https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-full-updated.png) 도 있습니다.

### Config

config 객체는 모델의 모든 하이퍼파라미터를 포함하고 있습니다. 참조 모델의 config를 출력하여 어떤 내용이 포함되어 있는지 확인할 수 있습니다:

In [ ]:
# As a reference - note there's a lot of stuff we don't care about in here, to do with library internals or other architectures
print(reference_gpt2.cfg)

우리는 모델을 위해 간소화된 config를 정의합니다:

In [ ]:
@dataclass
class Config:
    d_model: int = 768
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 50257
    init_range: float = 0.02
    n_ctx: int = 1024
    d_head: int = 64
    d_mlp: int = 3072
    n_heads: int = 12
    n_layers: int = 12


cfg = Config()
print(cfg)

### 테스트

테스트는 매우 유용합니다. 진행하면서 사용할 수 있도록 가벼운 테스트들을 작성해 보세요!

**단순 테스트(Naive test):** 적절한 shape의 랜덤 입력을 생성하여 모델에 입력하고, 에러가 발생하는지 확인한 뒤 올바른 출력을 출력합니다.

In [ ]:
def rand_float_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randn(shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")


def rand_int_test(cls, shape):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    random_input = t.randint(100, 1000, shape).to(device)
    print("Input shape:", random_input.shape)
    output = layer(random_input)
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape, "\n")


def load_gpt2_test(cls, gpt2_layer, input):
    cfg = Config(debug=True)
    layer = cls(cfg).to(device)
    layer.load_state_dict(gpt2_layer.state_dict(), strict=False)
    print("Input shape:", input.shape)
    orig_input = input.clone()
    output = layer(orig_input)
    assert t.allclose(input, orig_input), "Input has been modified, make sure operations are not done in place"
    if isinstance(output, tuple):
        output = output[0]
    print("Output shape:", output.shape)
    try:
        reference_output = gpt2_layer(input)
    except:
        reference_output = gpt2_layer(input, input, input)
    print("Reference output shape:", reference_output.shape, "\n")
    comparison = t.isclose(output, reference_output, atol=1e-4, rtol=1e-3)
    print(f"{comparison.sum() / comparison.numel():.2%} of the values are correct\n")
    assert 1 - (comparison.sum() / comparison.numel()) < 1e-5, "More than 0.01% of the values are incorrect"

### 연습 문제 - `LayerNorm` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

아래 코드를 작성한 후, 테스트를 실행하여 layer가 올바르게 작동하는지 확인하십시오.

작성하실 LayerNorm은 다음을 수행해야 합니다:

* 평균을 0으로 만듭니다.
* 분산이 1이 되도록 정규화합니다.
* 학습 가능한 weight로 스케일링합니다.
* 학습 가능한 bias로 이동(translate)시킵니다.

PyTorch [LayerNorm documentation](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)를 참고 자료로 사용할 수 있습니다. 몇 가지 추가 참고 사항은 다음과 같습니다:

* 작성하시는 layernorm 구현체는 항상 `affine=True`를 가집니다. 즉, 학습 파라미터인 `w`와 `b`를 학습합니다 (이는 PyTorch 문서에서 각각 $\gamma$와 $\beta$로 표현됩니다).
* 중심화(centering)와 정규화 이후, 입력의 길이가 `d_model`인 각 벡터는 평균 0, 분산 1을 가져야 함을 기억하십시오.
* PyTorch 문서 페이지에 명시된 대로, 분산은 `unbiased=False`를 사용하여 계산해야 합니다.
* config 객체의 `layer_norm_eps` 인자는 PyTorch 문서의 $\epsilon$ 항에 해당합니다 (이는 0으로 나누는 오류를 방지하기 위해 포함됩니다).
* config에 `debug` 인자를 제공했습니다. `debug=True`인 경우, 디버깅을 돕기 위해 `forward` 함수 내 객체의 shape와 같은 출력을 프린트할 수 있습니다 (이는 코딩 속도를 높이는 매우 유용한 팁입니다).

`raise NotImplementedError()`라고 표시된 부분의 함수를 완성하십시오 (이는 이 섹션의 대부분의 다른 연습 문제에서도 동일하게 적용되는 기본 패턴입니다).

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        raise NotImplementedError()


rand_float_test(LayerNorm, [2, 4, 768])
load_gpt2_test(LayerNorm, reference_gpt2.ln_final, cache["resid_post", 11])
tests.test_layer_norm_epsilon(LayerNorm, cache["resid_post", 11])

<details><summary>솔루션</summary>

```python
class LayerNorm(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, residual: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        residual_mean = residual.mean(dim=-1, keepdim=True)
        residual_std = (residual.var(dim=-1, keepdim=True, unbiased=False) + self.cfg.layer_norm_eps).sqrt()

        residual = (residual - residual_mean) / residual_std
        return residual * self.w + self.b
```
</details>

### 연습 문제 - `Embed` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

이것은 기본적으로 token에서 residual stream 벡터로 매핑되는 lookup table입니다.

(힌트 - 복잡한 함수 없이 단 한 줄로 구현할 수 있습니다. 만약 10분 이상 고민하고 계신다면, 아마 너무 어렵게 생각하고 계신 것일 겁니다!)

In [ ]:
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        raise NotImplementedError()


rand_int_test(Embed, [2, 4])
load_gpt2_test(Embed, reference_gpt2.embed, tokens)

<details>
<summary>도움말 - 계속해서 <code>RuntimeError: CUDA error: device-side assert triggered</code> 에러가 발생합니다.</summary>

이것은 매우 당혹스러운 유형의 에러 메시지입니다. 왜냐하면 (1) 커널을 재시작해야 하며, (2) 에러 메시지가 실제로 어디에서 발생했는지 알려주지 않는 경우가 많기 때문입니다!

두 번째 문제는 파일의 맨 윗부분(`os`을 import한 후)에 `os.environ['CUDA_LAUNCH_BLOCKING'] = "1"` 줄을 추가함으로써 해결할 수 있습니다. 이것이 버그 자체를 고쳐주지는 않지만, 정확한 발생 지점이 식별되도록 보장합니다.

버그를 실제로 수정하는 방법에 대해 말씀드리자면, 이 에러는 보통 잘못된 인덱싱의 결과로 발생합니다. 예를 들어, 최대 embedding 크기보다 큰 token들에 대해 embedding layer를 적용하려고 할 때 발생합니다.
</details>


<details><summary>해결책</summary>

```python
class Embed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_E = nn.Parameter(t.empty((cfg.d_vocab, cfg.d_model)))
        nn.init.normal_(self.W_E, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        return self.W_E[tokens]
```
</details>

### 연습 문제 - `PosEmbed` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

Positional embedding은 lookup table로 생각할 수도 있지만, 인덱스가 token ID인 대신 단순히 숫자 `0`, `1`, `2`, ..., `seq_len-1` (즉, 시퀀스 내 token의 위치 인덱스)가 인덱스가 됩니다.

In [ ]:
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        raise NotImplementedError()


rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)

<details><summary>솔루션</summary>

```python
class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_model"]:
        batch, seq_len = tokens.shape
        return einops.repeat(self.W_pos[:seq_len], "seq d_model -> batch seq d_model", batch=batch)
```
</details>

### 연습 문제 - `apply_causal_mask` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

causal mask 함수는 `Attention` 클래스의 메서드가 될 것입니다.
이 함수는 attention score를 입력으로 받아 마스크를 적용함으로써, 모델이 이전 위치에만 attention을 가질 수 있도록 합니다 (즉, 모델이 미래의 위치를 보고 답을 맞히는 부정행위를 할 수 없게 합니다).
우리는 이 함수를 먼저 구현하고 테스트한 뒤, `Attention` 클래스의 `forward` 메서드로 넘어갈 것입니다.

몇 가지 힌트입니다:

* attention score에 마스킹을 할 때 [`torch.where`](https://pytorch.org/docs/stable/generated/torch.where.html) 또는 [`torch.masked_fill_`](https://pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html) 함수를 사용할 수 있습니다.
* [`torch.triu`](https://pytorch.org/docs/stable/generated/torch.triu.html) 함수는 확률을 0으로 설정하려는 모든 위치에 대해 True 값을 가지는 마스크를 생성하는 데 유용합니다.
* 마스킹된 위치를 음의 무한대로 설정하기 위해 `self.IGNORE` 속성을 사용하십시오.
<details>
<summary>질문 - 왜 attention 확률을 0으로 설정하는 대신, attention score를 음의 무한대로 설정하여 마스킹한다고 생각하십니까?</summary>

만약 attention 확률에 마스킹을 한다면, 확률의 합이 더 이상 1이 되지 않을 것입니다.

우리는 score에 마스킹을 한 *다음*에 softmax를 취함으로써, 확률들이 여전히 유효한 확률(즉, 합이 1)이 되도록 하고, 마스킹된 위치의 값들이 모델의 출력에 아무런 영향을 주지 않도록 하고자 합니다.
</details>

In [ ]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def apply_causal_mask(
        self,
        attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"],
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        raise NotImplementedError()


tests.test_causal_mask(Attention.apply_causal_mask)

<details>
<summary>힌트 (의사코드)</summary>

```python
def apply_causal_mask(
    self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:

    # Define a mask that is True for all positions we want to set probabilities to zero for

    # Apply the mask to attention scores, then return the masked scores
```
</details>


<details><summary>솔루션</summary>

```python
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def apply_causal_mask(
        self,
        attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"],
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        # Define a mask that is True for all positions we want to set probabilities to zero for
        all_ones = t.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device)
        mask = t.triu(all_ones, diagonal=1).bool()
        # Apply the mask to attention scores, then return the masked scores
        attn_scores.masked_fill_(mask, self.IGNORE)
        return attn_scores
```
</details>

### 연습 문제 - `Attention` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 30-45 minutes on this exercise.
> ```

* **1단계:** attention pattern 생성 - 각 destination token에 대해, 이전 token들(현재 token 포함)에 대한 확률 분포를 계산합니다.
    * input -> query, key shape `[batch, seq_posn, head_index, d_head]` 로의 linear map을 적용합니다.
    * 모든 query와 key의 *쌍*에 대해 dot product를 수행하여 attn_scores `[batch, head_index, query_pos, key_pos]` 를 얻습니다 (query = dest, key = source).
    * **Scale**을 적용하고 mask `attn_scores` 를 통해 하삼각 행렬(lower triangular), 즉 causal하게 만듭니다.
    * `key_pos` 차원을 따라 softmax를 적용하여 각 query (destination) token에 대한 확률 분포를 얻습니다. 이것이 바로 attention pattern입니다!
* **2단계:** attention pattern을 사용하여 source token에서 destination token으로 정보를 이동시킵니다 (이동 = linear map 적용).
    * input -> value `[batch, key_pos, head_index, d_head]` 로의 linear map을 적용합니다.
    * attn pattern과 함께 `key_pos` 을 따라 mix하여 `z` 을 얻으며, 이는 value 벡터들의 가중 평균 `[batch, query_pos, head_index, d_head]` 입니다.
    * output으로 맵핑합니다, `[batch, position, d_model]` (position = query_pos, 모든 head에 대해 합산한 결과입니다).

참고 - **scale**이라고 말하는 것은 `sqrt(d_head)` 로 나누는 것을 의미합니다. 이것의 목적은 vanishing gradients를 방지하는 것입니다 (이는 softmax와 같은 함수를 다룰 때 큰 문제인데, 만약 하나의 값이 다른 모든 값보다 훨씬 크다면 확률이 0 또는 1에 가까워지고 gradient가 0에 가까워지기 때문입니다).

아래는 앞서 보았던 attention head 다이어그램의 훨씬 더 크고 상세한 버전입니다. 이를 통해 실제 어떤 tensor 연산들이 포함되는지 파악하실 수 있을 것입니다. 이 다이어그램에 대한 몇 가지 설명은 다음과 같습니다:

* 그림에 세 번째 차원이 표시될 때마다, 이는 `head_index` 차원을 의미합니다. attention layer 내의 모든 연산이 각 head에 대해 독립적으로 수행됨을 알 수 있습니다.
* 박스 안의 객체들은 activation입니다. 이들은 batch 차원을 가집니다 (단순화를 위해 다이어그램에서는 batch 차원이 1이라고 가정합니다). 박스 오른쪽의 객체들은 parameter (weights 및 biases)이며, batch 차원이 없습니다.
* key, query, value를 `(batch, seq_pos, head_idx, d_head)` 로 배치하는데, 이는 bias의 shape가 `(head_idx, d_head)` 이므로 bias를 더하기에 편리하기 때문입니다 (array broadcasting 규칙을 기억하십시오!).

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/main/img/transformer-attn-30.png" width="1400">

<details>
<summary><b>attention에 관한 몇 가지 추가 참고 사항 (선택 사항)</b></summary>

<!-- Usually we have the relation `e = n * h` (i.e. `d_model = num_heads * d_head`). There are some computational justifications for this, but mostly this is just done out of convention (just like how we usually have `d_mlp = 4 * d_model`!). -->

여기서는 attention head의 수학적 공식(특히 **QK** 및 **OV** circuit의 분리)과 관련된 몇 가지 세부 사항을 다룹니다. 이는 이 장의 다음 연습 문제 세트에서 훨씬 더 깊게 다룰 내용입니다.

**QK** circuit은 $W_Q$ 및 $W_K$ 행렬의 연산으로 구성됩니다. 다시 말해, 이는 attention pattern, 즉 residual stream에서 정보가 어디로 이동하고 어디서 오는지 결정합니다. attention pattern $A$의 함수 형태는 다음과 같습니다:

$$
A = \text{softmax}\left(\frac{x W_Q W_K^T x^T}{\sqrt{d_{head}}}\right)
$$

여기서 $x$은 residual stream(shape `[seq_len, d_model]`)이며, $W_Q$, $W_K$은 단일 head의 가중치 행렬(즉, shape `[d_model, d_head]`)입니다.

**OV** circuit은 $W_V$ 및 $W_O$ 행렬의 연산으로 구성됩니다. attention pattern이 고정되면, 이 행렬들은 소스 위치의 residual stream에 작용하며, 그 출력값이 소스에서 목적지 위치로 이동하는 대상이 됩니다.

아래 다이어그램은 OV circuit의 함수 형태를 보여줍니다. QK circuit(분홍색)은 목적지 token이 소스 token에 attend하도록 만드는 역할을 하며, OV circuit(연갈색)은 소스 token 데이터를 목적지 token으로 보낼 정보로 실제로 매핑하는 역할을 합니다.

<img src="https://raw.githubusercontent.com/chloeli-15/ARENA_img/refs/heads/main/img/qkv.png" width="800">

전체 attention head의 함수 형태는 다음과 같습니다:

$$
\begin{aligned}
\text{output} &= \text{softmax}\left(\frac{x W_Q W_K^T x^T}{\sqrt{d_{head}}}\right) (x W_V W_O) \\
    &= Ax W_V W_O
\end{aligned}
$$

여기서 $W_V$의 shape은 `[d_model, d_head]`이며, $W_O$의 shape은 `[d_head, d_model]`입니다.

여기서 우리는 **QK circuit**과 **OV circuit**이 개념적으로 서로 다른 일을 하고 있으며, attention head의 두 개의 별개 부분으로 생각해야 한다는 것을 명확히 알 수 있습니다.

다시 한번 말씀드리지만, 지금 당장 이 모든 내용을 이해하지 못하더라도 걱정하지 마십시오. 이후의 연습 문제에서 이 모든 내용에 대해 **훨씬** 더 자세히 다룰 예정입니다. 여기서 이 논의를 하는 목적은 앞으로 나올 내용이 어떤 것인지 맛보게 해드리기 위함입니다!

</details>

먼저, attention pattern을 시각화하고 살펴보는 것이 유용합니다. 여기서 우리가 정확히 무엇을 보고 있는 것일까요? (특정 head를 클릭하면 해당 head의 pattern만 고정해서 보여주므로, 해석하기가 더 쉬워집니다.)

In [ ]:
import circuitsvis as cv
from IPython.display import display

display(
    cv.attention.attention_patterns(
        tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache["pattern", 0][0]
    )
)

데이터를 다른 방식으로 보여주는 `attention_heads` 함수를 사용할 수도 있습니다 (구문은 `attention_patterns`과 완전히 동일합니다). VSCode에서 이를 표시할 경우 메인 plot의 크기가 계속해서 줄어드는 버그가 발생할 수 있으니 주의하시기 바랍니다. 이런 현상이 발생하면 대신 HTML로 저장하여(즉, `html = cv.attention.attention_heads(...); with open("attn_heads.html", "w") as f: f.write(str(html))`를 사용하여) 브라우저에서 plot을 여시기 바랍니다.

<!-- <details>
<summary>도움말 - <code>attention_heads</code> plot이 이상하게 작동합니다.</summary>

이는 `circuitsvis`의 버그로 보입니다. VSCode에서 attention head plot의 크기가 계속해서 줄어듭니다.

이 문제가 해결될 때까지, 이를 우회하는 한 가지 방법은 브라우저에서 plot을 여는 것입니다. `webbrowser` 라이브러리를 사용하여 인라인으로 수행할 수 있습니다:

```python
attn_heads = cv.attention.attention_heads(
    tokens=reference_gpt2.to_str_tokens(reference_text),
    attention=cache["pattern", 0][0]
)

path = "attn_heads.html"

with open(path, "w") as f:
    f.write(str(attn_heads))

webbrowser.open(path)
```

정확히 어디에 저장되고 있는지 확인하려면 `os.getcwd()`을 사용하여 현재 작업 디렉토리를 출력하면 됩니다.
</details> -->

In [ ]:
display(
    cv.attention.attention_heads(
        tokens=reference_gpt2.to_str_tokens(reference_text), attention=cache["pattern", 0][0]
    )
)

아래 `Attention`의 forward 메서드를 작성해야 합니다. 또한 `apply_causal_mask`에서 작성한 코드를 `Attention`의 새로운 구현체로 복사하십시오 (기존 구현 코드의 나머지 부분은 삭제해도 됩니다).

참고로, 이 구현은 아마도 이 페이지에서 가장 어려운 연습 문제가 될 것이므로, 시간이 좀 걸리더라도 걱정하지 마십시오! 막히는 부분이 있다면 솔루션의 일부를 참고하시기 바랍니다. 몇 가지 팁은 다음과 같습니다:

* attention score scaling을 잊지 마십시오 (이는 masking 이전에 수행되어야 합니다).
* 너무 많은 연산을 한 줄의 코드로 결합하지 않도록 노력하십시오.
* 변수 이름을 설명적으로 작성하십시오 (즉, 단순히 `x = some_fn_of(x), x = some_other_fn_of(x), ...`과 같이 짓지 마십시오).

In [ ]:
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        raise NotImplementedError()

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        # You should copy your solution from earlier
        raise NotImplementedError()


tests.test_causal_mask(Attention.apply_causal_mask)
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["normalized", 0, "ln1"])

<details>
<summary>힌트 (forward 메서드를 위한 의사코드)</summary>

```python
def forward(
    self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]
) -> Float[Tensor, "batch posn d_model"]:

    # Calculate query, key and value vectors
    q, k, v = ...

    # Calculate attention scores, then scale and mask, and apply softmax to get probabilities
    attn_scores = ...
    attn_scores_masked = ...
    attn_pattern = ...

    # Take weighted sum of value vectors, according to attention probabilities
    z = ...

    # Calculate output (by applying matrix W_O and summing over heads, then adding bias b_O)
    attn_out = ...
    return attn_out
```
</details>


<details><summary>솔루션</summary>

```python
class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(float("-inf"), dtype=t.float32, device=device))

    def forward(self, normalized_resid_pre: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        # Calculate query, key and value vectors
        q = (
            einops.einsum(
                normalized_resid_pre,
                self.W_Q,
                "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
            )
            + self.b_Q
        )
        k = (
            einops.einsum(
                normalized_resid_pre,
                self.W_K,
                "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
            )
            + self.b_K
        )
        v = (
            einops.einsum(
                normalized_resid_pre,
                self.W_V,
                "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head",
            )
            + self.b_V
        )

        # Calculate attention scores, then scale and mask, and apply softmax to get probabilities
        attn_scores = einops.einsum(
            q,
            k,
            "batch posn_Q nheads d_head, batch posn_K nheads d_head -> batch nheads posn_Q posn_K",
        )
        attn_scores_masked = self.apply_causal_mask(attn_scores / self.cfg.d_head**0.5)
        attn_pattern = attn_scores_masked.softmax(-1)

        # Take weighted sum of value vectors, according to attention probabilities
        z = einops.einsum(
            v,
            attn_pattern,
            "batch posn_K nheads d_head, batch nheads posn_Q posn_K -> batch posn_Q nheads d_head",
        )

        # Calculate output (by applying matrix W_O and summing over heads, then adding bias b_O)
        attn_out = (
            einops.einsum(
                z,
                self.W_O,
                "batch posn_Q nheads d_head, nheads d_head d_model -> batch posn_Q d_model",
            )
            + self.b_O
        )

        return attn_out

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        """
        Applies a causal mask to attention scores, and returns masked scores.
        """
        # Define a mask that is True for all positions we want to set probabilities to zero for
        all_ones = t.ones(attn_scores.size(-2), attn_scores.size(-1), device=attn_scores.device)
        mask = t.triu(all_ones, diagonal=1).bool()
        # Apply the mask to attention scores, then return the masked scores
        attn_scores.masked_fill_(mask, self.IGNORE)
        return attn_scores
```
</details>

### 연습 문제 - `MLP` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

다음으로, 다음과 같이 구성된 MLP layer를 구현해야 합니다:

* weight `W_in`, bias `b_in`를 가진 linear layer
* 비선형 함수 (일반적으로 GELU를 사용하며, 이를 위해 `gelu_new` 함수가 임포트되었습니다)
* weight `W_out`, bias `b_out`를 가진 linear layer

In [ ]:
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        raise NotImplementedError()


rand_float_test(MLP, [2, 4, 768])
load_gpt2_test(MLP, reference_gpt2.blocks[0].mlp, cache["normalized", 0, "ln2"])

<details><summary>솔루션</summary>

```python
class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_in = nn.Parameter(t.empty((cfg.d_model, cfg.d_mlp)))
        self.W_out = nn.Parameter(t.empty((cfg.d_mlp, cfg.d_model)))
        self.b_in = nn.Parameter(t.zeros((cfg.d_mlp)))
        self.b_out = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_in, std=self.cfg.init_range)
        nn.init.normal_(self.W_out, std=self.cfg.init_range)

    def forward(self, normalized_resid_mid: Float[Tensor, "batch posn d_model"]) -> Float[Tensor, "batch posn d_model"]:
        pre = (
            einops.einsum(
                normalized_resid_mid,
                self.W_in,
                "batch position d_model, d_model d_mlp -> batch position d_mlp",
            )
            + self.b_in
        )
        post = gelu_new(pre)
        mlp_out = (
            einops.einsum(post, self.W_out, "batch position d_mlp, d_mlp d_model -> batch position d_model")
            + self.b_out
        )
        return mlp_out
```
</details>

### 연습 문제 - `TransformerBlock` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

이제 attention, MLP, 그리고 layernorm을 하나의 transformer block으로 합칠 수 있습니다. residual connection을 정확하게 구현하는 것을 잊지 마십시오!

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
        raise NotImplementedError()


rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])

<details>
<summary>도움말 - 이전까지의 모든 모듈에서는 100% 정확도가 나왔는데, 이 모듈에서만 약 90%의 정확도가 나옵니다.</summary>

이는 layernorm 구현 방식이 `(var + eps).sqrt()` 대신 `std + eps`으로 나누기 때문일 수 있습니다. 후자가 GPT-2에서 사용되는 구현 방식과 일치하며, 이 오류는 해당 테스트에서만 나타납니다.

</details>


<details><summary>솔루션</summary>

```python
class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(self, resid_pre: Float[Tensor, "batch position d_model"]) -> Float[Tensor, "batch position d_model"]:
        resid_mid = self.attn(self.ln1(resid_pre)) + resid_pre
        resid_post = self.mlp(self.ln2(resid_mid)) + resid_mid
        return resid_post
```
</details>

### 연습 문제 - `Unembed` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to ~10 minutes on this exercise.
> ```

unembedding은 단순히 linear layer(weight `W_U` 및 bias `b_U` 포함)입니다.

In [ ]:
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab), requires_grad=False))

    def forward(
        self, normalized_resid_final: Float[Tensor, "batch position d_model"]
    ) -> Float[Tensor, "batch position d_vocab"]:
        raise NotImplementedError()


rand_float_test(Unembed, [2, 4, 768])
load_gpt2_test(Unembed, reference_gpt2.unembed, cache["ln_final.hook_normalized"])

<details><summary>솔루션</summary>

```python
class Unembed(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.W_U = nn.Parameter(t.empty((cfg.d_model, cfg.d_vocab)))
        nn.init.normal_(self.W_U, std=self.cfg.init_range)
        self.b_U = nn.Parameter(t.zeros((cfg.d_vocab), requires_grad=False))

    def forward(
        self, normalized_resid_final: Float[Tensor, "batch position d_model"]
    ) -> Float[Tensor, "batch position d_vocab"]:
        return (
            einops.einsum(
                normalized_resid_final,
                self.W_U,
                "batch posn d_model, d_model d_vocab -> batch posn d_vocab",
            )
            + self.b_U
        )
```
</details>

### 연습 문제 - `DemoTransformer` 구현하기

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

In [ ]:
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
        raise NotImplementedError()


rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)

<details><summary>솔루션</summary>

```python
class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
        residual = self.embed(tokens) + self.pos_embed(tokens)
        for block in self.blocks:
            residual = block(residual)
        logits = self.unembed(self.ln_final(residual))
        return logits
```
</details>

**직접 시도해 보세요!**

In [ ]:
demo_gpt2 = DemoTransformer(Config(debug=False)).to(device)
demo_gpt2.load_state_dict(reference_gpt2.state_dict(), strict=False)

demo_logits = demo_gpt2(tokens)

테스트 문자열을 사용하여 loss를 계산해 보겠습니다!

우리는 **cross-entropy loss** 공식을 사용합니다. 모델링된 분포 $Q$ 와 타겟 분포 $P$ 사이의 cross entropy loss는 다음과 같습니다:

$$
-\sum_x P(x) \log Q(x)
$$

$P$ 이 단순히 타겟 클래스의 경험적 분포인 경우(즉, 정답 클래스 $x^*$ 에 대해 $P(x^*) = 1$ 인 경우), 이는 다음과 같이 됩니다:

$$
-\log Q(x^*)
$$

다시 말해, 정답 분류에 대한 negative log prob가 됩니다.

In [ ]:
def get_log_probs(
    logits: Float[Tensor, "batch posn d_vocab"], tokens: Int[Tensor, "batch posn"]
) -> Float[Tensor, "batch posn-1"]:
    log_probs = logits.log_softmax(dim=-1)
    # Get logprobs the first seq_len-1 predictions (so we can compare them with the actual next tokens)
    log_probs_for_tokens = log_probs[:, :-1].gather(dim=-1, index=tokens[:, 1:].unsqueeze(-1)).squeeze(-1)

    return log_probs_for_tokens


pred_log_probs = get_log_probs(demo_logits, tokens)
print(f"Avg cross entropy loss: {-pred_log_probs.mean():.4f}")
print(f"Avg cross entropy loss for uniform distribution: {math.log(demo_gpt2.cfg.d_vocab):4f}")
print(f"Avg probability assigned to correct token: {pred_log_probs.exp().mean():4f}")

가장 가능성이 높은 다음 token을 선택하고, 이를 모델에 다시 입력하기 전에 prompt에 계속해서 추가함으로써 탐욕적(greedily)으로 텍스트를 생성할 수도 있습니다:

In [ ]:
test_string = """Mitigating the risk of extinction from AI should be a global priority alongside other societal-scale risks such as"""
for i in tqdm(range(100)):
    test_tokens = reference_gpt2.to_tokens(test_string).to(device)
    demo_logits = demo_gpt2(test_tokens)
    test_string += reference_gpt2.tokenizer.decode(demo_logits[-1, -1].argmax())

print(test_string)

4️⃣ 섹션에서는 단순히 출력값에 argmax를 취하는 것보다 조금 더 흥미로운 방식으로 텍스트를 생성하는 방법을 배울 것입니다 (단순 argmax는 반복과 같은 부자연스러운 패턴이나, 덜 자연스럽게 들리는 텍스트로 이어질 수 있습니다).

# 3️⃣ Transformer 학습시키기

> ##### 학습 목표
>
> * transformer를 처음부터 학습시키는 방법을 이해합니다.
> * 기본적인 transformer 학습 루프를 작성합니다.
> * 학습 데이터의 특징(예: bigram 빈도)과 관련하여 transformer의 cross entropy loss가 감소하는 과정을 해석합니다.

이제 transformer를 구축했고, 가중치를 로드했을 때 예상대로 작동하는 것을 확인했으므로, 처음부터 직접 학습시켜 보겠습니다!

이 과정은 이 코드를 사용하여 실제로 자신만의 GPT-2를 어떻게 학습시킬 수 있는지 보여주는 가벼운 데모입니다! 여기서는 아주 작은 데이터셋으로 아주 작은 모델을 학습시키지만, 더 크고 실제적인 모델을 학습시키는 코드와 근본적으로 동일합니다 (다만, 효율적으로 수행하려면 더 강력한 GPU와 data parallelism이 필요하며, 훨씬 더 큰 모델의 경우 더 정교한 parallelism이 필요합니다).

학습 과정이 어떻게 진행되는지 보여주기 위해 (그리고 노트북이 colab이나 사용자의 컴퓨터를 멈추게 하지 않기 위해), context length 128, 레이어당 16개의 head를 가진 4 레이어 모델을 batch size 32로 10*500 step 동안 학습시키겠습니다.

## 모델 생성

In [ ]:
model_cfg = Config(
    debug=False,
    d_model=32,
    n_heads=16,
    d_head=2,
    d_mlp=32 * 4,
    n_layers=4,
    n_ctx=128,
    d_vocab=reference_gpt2.cfg.d_vocab,
)
model = DemoTransformer(model_cfg)

## Training Args


참고로, 이번 최적화를 위해 **weight decay**를 사용할 예정입니다.

In [ ]:
@dataclass
class TransformerTrainingArgs:
    batch_size: int = 32
    epochs: int = 10
    max_steps_per_epoch: int = 500
    lr: float = 1e-3
    weight_decay: float = 1e-2
    wandb_project: str | None = "day1-demotransformer"
    wandb_name: str | None = None


args = TransformerTrainingArgs()

## 데이터 생성

우리는 일반적인 3~4세 아이들이 이해할 수 있는 작은 단어 집합만을 사용하여 합성 생성된 간단한 이야기 데이터셋인 [TinyStories dataset](https://huggingface.co/datasets/roneneldan/TinyStories)를 로드합니다. 이 데이터셋은 여전히 일관된 텍스트를 생성할 수 있는 [exploring how small a LLM can be](https://arxiv.org/pdf/2305.07759)를 위해 설계되었습니다.

In [ ]:
dataset = datasets.load_dataset("roneneldan/TinyStories", split="train")
print(dataset)
print(dataset[0]["text"])

`tokenize_and_concatenate`은 문자열 데이터셋을 입력받아 모델에 입력할 준비가 된 token ID 데이터셋을 반환하는 유용한 함수입니다. 그런 다음 이 tokenized 데이터셋으로부터 dataloader를 생성합니다. 유용한 메서드인 `train_test_split`을 통해 training 세트와 testing 세트를 얻을 수 있습니다.

In [ ]:
tokenized_dataset = tokenize_and_concatenate(
    dataset,
    reference_gpt2.tokenizer,
    streaming=False,
    max_length=model.cfg.n_ctx,
    column_name="text",
    add_bos_token=True,
    num_proc=4,
)

dataset_dict = tokenized_dataset.train_test_split(test_size=1000)
train_loader = DataLoader(
    dataset_dict["train"], batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=True
)
test_loader = DataLoader(
    dataset_dict["test"], batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True
)

이 dataloader들을 통해 반복문을 실행하면, `(batch, seq_len)` 형상을 가진 token ID tensor로 매핑되는 단일 키 `'tokens'`를 가진 dictionary들을 찾게 됩니다.

In [ ]:
first_batch = train_loader.dataset[: args.batch_size]

print(first_batch.keys())
print(first_batch["tokens"].shape)

## Training Loop

첫 번째 주에 [training loops](https://arena-ch0-fundamentals.streamlit.app/[0.3]_ResNets#training-loop) 내용을 학습하셨다면, 이 모든 내용이 익숙하실 것입니다. 그렇지 않다면, 해당 섹션을 빠르게 훑어보며 핵심 개념에 대한 개요를 파악하시기 바랍니다. **Training loop** 섹션의 시작 부분이 가장 중요하며, [Modularisation](https://arena-ch0-fundamentals.streamlit.app/[0.3]_ResNets#modularisation) 및 [dataclasses](https://arena-ch0-fundamentals.streamlit.app/[0.3]_ResNets#aside-dataclasses)에 관한 하위 섹션들도 매우 유용합니다. 마지막으로, 모델 학습을 위해 Weights and Biases를 사용할 예정이며, 사용 방법은 [here](https://arena-ch0-fundamentals.streamlit.app/[0.4]_Optimization#what-is-weights-and-biases)에서 읽어보실 수 있습니다. 다음 실습을 위해 알고 있어야 할 (대략적인) 모든 내용은 다음과 같습니다:

* gradient update 단계의 핵심 부분은 다음과 같습니다:
    * 모델의 출력과 실제 레이블 사이의 (cross-entropy) loss 계산,
    * `loss.backward()` - 모델 파라미터에 대한 loss의 gradient 계산,
    * `optimizer.step()` - gradient를 사용하여 모델 파라미터 업데이트,
    * `optimizer.zero_grad()` - gradient가 누적되지 않도록 zero 처리.
* training loop를 클래스로 깔끔하게 패키징할 수 있으며, 여기에는 학습 및 validation 단계를 위한 메서드 등이 포함됩니다. 이는 다양한 컨텍스트에서 재사용 가능한 코드를 작성하는 데 도움이 됩니다.
* dataclasses를 사용하여 학습과 관련된 모든 인자를 한곳에 저장하고, 이를 trainer 클래스에 전달할 수 있습니다. 자동 완성 기능이 제공된다는 점이 큰 장점입니다!
    * 이때 scope에 주의하십시오. trainer 클래스 내에서 전역 `args`이 아닌 `self.args`을 참조하고 있는지 확인해야 합니다.
* Weights and Biases를 사용하여 실험을 추적하고 관련 변수를 로그로 남길 수 있습니다. 세 가지 필수 함수는 다음과 같습니다:
    * `wandb.init()` - 새로운 run을 초기화하며, `project`, `name`, `config` 등의 인자를 받습니다.
    * `wandb.log()` - 변수 딕셔너리를 로그로 남깁니다 (예: `{"loss": loss}`). 또한 `step` 인자를 받습니다.
    * `wandb.finish()` - 학습 종료 시 호출됩니다 (인자 없음).

### 연습 문제 - 학습 루프 작성하기

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵⚪
> 
> You should spend up to 10-20 minutes on this exercise.
> ```

아래의 메서드들을 완성해야 합니다. 몇 가지 안내 사항입니다:

* 이전 섹션에서 `get_log_probs` 함수를 사용하여 cross entropy loss를 계산할 수 있었음을 기억하십시오.
* optimizer `t.optim.AdamW` (weight decay가 적용된 Adam)를 사용해야 하며, 하이퍼파라미터 `lr` 및 `weight_decay`는 `TransformerTrainingArgs` dataclass 인스턴스에서 가져와 사용하십시오.
* 각 epoch의 학습 단계가 너무 오래 지속되지 않도록 하기 위한 임시 방편인 인자 `max_steps_per_epoch`을 제공했습니다. 이 step 수만큼 학습 단계가 진행된 후 종료하면 됩니다. 기본값은 모델의 유의미한 성능을 보여줄 수 있을 만큼 매우 짧게 실행되도록 설정되어 있습니다.
* `tokens.to(device)` (노트북 상단에 정의된 전역 변수여야 합니다)을 통해 token들을 device로 이동시키는 것을 잊지 마십시오.
* 원하신다면 [previous chapter of the course](https://arena-ch0-fundamentals.streamlit.app/[0.3]_ResNets#training-loop)의 학습 루프를 다시 참고하셔도 좋습니다.
* 학습 중에 모델이 어떻게 작동하는지 확인하기 위해 텍스트를 생성할 수 있도록 `TransformerSampler` 클래스의 인스턴스도 제공했습니다. sampling이 어떻게 작동하는지는 다음 섹션에서 다루겠습니다.

In [ ]:
class TransformerTrainer:
    def __init__(self, args: TransformerTrainingArgs, model: DemoTransformer):
        super().__init__()
        self.model = model
        self.args = args
        self.sampler = solutions.TransformerSampler(self.model, reference_gpt2.tokenizer)
        self.optimizer = t.optim.AdamW(self.model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        self.step = 0

        self.train_loader = DataLoader(
            dataset_dict["train"],
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
        )
        self.test_loader = DataLoader(
            dataset_dict["test"],
            batch_size=args.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

    def training_step(self, batch: dict[str, Int[Tensor, "batch seq"]]) -> Float[Tensor, ""]:
        """
        Calculates the loss on the tokens in the batch, performs a gradient update step, and logs the loss.

        Remember that `batch` is a dictionary with the single key 'tokens'.
        """
        raise NotImplementedError()
        return loss

    @t.inference_mode()
    def evaluate(self) -> float:
        """
        Evaluate the model on the test set and return the accuracy.
        """
        self.model.eval()
        #
        # YOUR CODE HERE - fill in the `evaluate` method
        #
        self.model.train()
        return accuracy

    def train(self):
        """
        Trains the model, for `self.args.epochs` epochs. Also handles wandb initialisation, and early stopping
        for each epoch at `self.args.max_steps_per_epoch` steps.
        """
        wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
        accuracy = np.nan

        progress_bar = tqdm(total=self.args.max_steps_per_epoch * self.args.epochs)

        for epoch in range(self.args.epochs):
            for i, batch in enumerate(self.train_loader):
                loss = self.training_step(batch)
                progress_bar.update()
                progress_bar.set_description(f"Epoch {epoch + 1}, loss: {loss:.3f}, accuracy: {accuracy:.3f}")
                if i >= self.args.max_steps_per_epoch:
                    break

            accuracy = self.evaluate()
            sample_text = self.sampler.sample("Once upon a time", max_tokens_generated=50)
            print(sample_text)

        wandb.finish()


# See the full run here: https://api.wandb.ai/links/dquarel/nrxuwnv7
model = DemoTransformer(model_cfg).to(device)
args = TransformerTrainingArgs()
trainer = TransformerTrainer(args, model)
trainer.train()

<details><summary>솔루션</summary>

```python
class TransformerTrainer:
    def __init__(self, args: TransformerTrainingArgs, model: DemoTransformer):
        super().__init__()
        self.model = model
        self.args = args
        self.sampler = solutions.TransformerSampler(self.model, reference_gpt2.tokenizer)
        self.optimizer = t.optim.AdamW(self.model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        self.step = 0

        self.train_loader = DataLoader(
            dataset_dict["train"],
            batch_size=args.batch_size,
            shuffle=True,
            num_workers=4,
            pin_memory=True,
        )
        self.test_loader = DataLoader(
            dataset_dict["test"],
            batch_size=args.batch_size,
            shuffle=False,
            num_workers=4,
            pin_memory=True,
        )

    def training_step(self, batch: dict[str, Int[Tensor, "batch seq"]]) -> Float[Tensor, ""]:
        """
        Calculates the loss on the tokens in the batch, performs a gradient update step, and logs the loss.

        Remember that `batch` is a dictionary with the single key 'tokens'.
        """
        tokens = batch["tokens"].to(device)
        logits = self.model(tokens)
        loss = -get_log_probs(logits, tokens).mean()
        loss.backward()
        self.optimizer.step()
        self.optimizer.zero_grad()
        self.step += 1
        wandb.log({"train_loss": loss}, step=self.step)
        return loss

    @t.inference_mode()
    def evaluate(self) -> float:
        """
        Evaluate the model on the test set and return the accuracy.
        """
        self.model.eval()
        total_correct, total_samples = 0, 0

        for batch in tqdm(self.test_loader, desc="Evaluating"):
            tokens = batch["tokens"].to(device)
            logits: Tensor = self.model(tokens)[:, :-1]
            predicted_tokens = logits.argmax(dim=-1)
            total_correct += (predicted_tokens == tokens[:, 1:]).sum().item()
            total_samples += tokens.size(0) * (tokens.size(1) - 1)

        accuracy = total_correct / total_samples
        wandb.log({"accuracy": accuracy}, step=self.step)
        self.model.train()
        return accuracy

    def train(self):
        """
        Trains the model, for `self.args.epochs` epochs. Also handles wandb initialisation, and early stopping
        for each epoch at `self.args.max_steps_per_epoch` steps.
        """
        wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
        accuracy = np.nan

        progress_bar = tqdm(total=self.args.max_steps_per_epoch * self.args.epochs)

        for epoch in range(self.args.epochs):
            for i, batch in enumerate(self.train_loader):
                loss = self.training_step(batch)
                progress_bar.update()
                progress_bar.set_description(f"Epoch {epoch + 1}, loss: {loss:.3f}, accuracy: {accuracy:.3f}")
                if i >= self.args.max_steps_per_epoch:
                    break

            accuracy = self.evaluate()
            sample_text = self.sampler.sample("Once upon a time", max_tokens_generated=50)
            print(sample_text)

        wandb.finish()
```
</details>

<!-- Note - this section of the course used to use PyTorch Lightning, but this has now been taken out. If you want, you can look at the old version of the training code which used PyTorch Lightning in the dropdown below.

<details>
<summary>PyTorch Lighting 학습 루프</summary>

```python
class LitTransformer(pl.LightningModule):
	def __init__(self, args: TransformerTrainingArgs, model: DemoTransformer, data_loader: DataLoader):
		super().__init__()
		self.model = model
		self.cfg = model.cfg
		self.args = args
		self.data_loader = data_loader

	def forward(self, tokens: Int[Tensor, "batch position"]) -> Float[Tensor, "batch position d_vocab"]:
		logits = self.model(tokens)
		return logits

	def training_step(self, batch: Dict[str, Tensor], batch_idx: int) -> Float[Tensor, ""]:
		'''
		Here you compute and return the training loss and some additional metrics for e.g.
		the progress bar or logger.
		'''
		tokens = batch["tokens"].to(device)
		logits = self.model(tokens)
		loss = -get_log_probs(logits, tokens).mean()
		self.log("train_loss", loss)
		return loss

	def configure_optimizers(self):
		'''
		Choose what optimizers and learning-rate schedulers to use in your optimization.
		'''
		optimizer = t.optim.AdamW(self.model.parameters(), lr=self.args.lr, weight_decay=self.args.weight_decay)
		return optimizer

	def train_dataloader(self):
		return self.data_loader


litmodel = LitTransformer(args, model, data_loader)
logger = WandbLogger(save_dir=args.log_dir, project=args.log_name, name=args.run_name)

trainer = pl.Trainer(
    max_epochs=args.max_epochs,
    logger=logger,
    log_every_n_steps=args.log_every_n_steps
)
trainer.fit(model=litmodel, train_dataloaders=litmodel.data_loader)
wandb.finish()
```

</details>

<details>
<summary>PyTorch Lightning을 더 이상 사용하지 않는 이유에 대한 설명</summary>

요약하자면 - PyTorch Lightning은 훌륭한 모듈화와 코드 저장 기능을 제공하지만, 학습 루프의 많은 세부 사항을 추상화하기 때문에 교육적인 목적으로는 그리 유용하지 않습니다. 또한, 학습 루프가 작동하는 방식에 대해 많은 구조를 강제하며 유연성이 부족합니다. 따라서 우리가 나중에 작성할 많은 코드(예: linear probes 또는 RL)가 이 프레임워크에 잘 맞지 않습니다. 하지만 기본 개념을 익힌 후 제공되는 다양한 추가 기능의 이점을 얻고자 할 때는 매우 유용한 도구가 될 수 있습니다.

</details> -->

코드를 처음 실행할 때, Weights and Biases에 로그인하고 VSCode에 API 키를 붙여넣어야 합니다. 이 작업이 완료되면 Weights and Biases 학습 실행이 시작됩니다. 많은 출력 텍스트가 나타나며, 그중 한 줄은 다음과 같이 보일 것입니다:

```
View run at https://wandb.ai/<USERNAME>/<PROJECT-NAME>/runs/<RUN-NAME>
```

이 링크를 클릭하면 실행 페이지를 방문할 수 있습니다.

> 참고 - Weights and Biases에서 그래프를 더 명확하게 보려면, 그래프의 **edit panel**(우측 상단의 작은 연필 모양 아이콘)을 클릭한 다음 **smoothing** 슬라이더를 오른쪽으로 이동하면 됩니다.

### 이 loss curve에 관한 참고 사항 (선택 사항)


우리 loss curve의 모양이 왜 이럴까요? 약 10-11에서 시작하여 매우 빠르게 떨어지다가, 이후 평탄해지는 것처럼 보입니다. 알고 보니, 이는 모델이 학습 과정에서 배우는 알고리즘의 종류와 관련이 있습니다.

처음에 모델은 랜덤 노이즈를 출력하며, 이는 "각 token을 거의 균등한 확률로 예측"하는 것과 비슷하게 보일 수 있습니다. 즉, 모든 $x$에 대해 $Q(x) = 1/d_\text{vocab}$ 입니다. 이는 우리에게 $\log (d_\text{vocab})$ 의 cross entropy loss를 줍니다.

In [ ]:
d_vocab = model.cfg.d_vocab

print(f"d_vocab = {d_vocab}")
print(f"Cross entropy loss on uniform distribution = {math.log(d_vocab):.3f}")

모델이 다음에 학습할 것으로 기대할 수 있는 것은 영어 단어의 빈도수입니다. 결국 `" and"` 또는 `" the"`과 같이 작고 흔한 token들은 다른 token들보다 훨씬 더 자주 나타날 수 있습니다. 이는 다음과 같은 평균 cross entropy loss를 제공할 것입니다:

$$
- \sum_x p_x \log p_x
$$

여기서 $p_x$은 학습 데이터에서 해당 단어의 실제 빈도수입니다.

우리는 이 값을 다음과 같이 평가할 수 있습니다:

In [ ]:
toks = tokenized_dataset[:]["tokens"].flatten()

d_vocab = model.cfg.d_vocab
freqs = t.bincount(toks, minlength=d_vocab)
probs = freqs.float() / freqs.sum()

distn = t.distributions.categorical.Categorical(probs=probs)
entropy = distn.entropy()

print(f"Entropy of training data = {entropy:.3f}")

unigram 빈도 이후에 모델이 보통으로 학습하는 다음 내용은 **bigram 빈도**(즉, 학습 데이터에서 인접한 token 쌍의 빈도)입니다. 예를 들어, `"I"`과 `" am"`은 흔한 token들이지만, 이들의 bigram 빈도는 각각 독립적으로 발생했을 때보다 훨씬 더 높습니다. bigram 빈도는 다음과 같은 사항들에도 도움이 되기 때문에 실제로 상당히 많은 부분을 해결해 줍니다:

* 몇 가지 간단한 문법 규칙 (예: 마침표 뒤에 대문자로 시작하는 단어가 오는 경우)
* tokenization의 특이한 점들 (예: `" manip"` 뒤에 `"ulative"`이 오는 경우)
* 흔한 이름들 (예: `"Barack"` 뒤에 `" Obama"`가 오는 경우)

bigram 빈도를 근사한 후에는 trigram(attention head를 사용해서만 구현 가능), **induction heads**(다음 연습 문제 세트에서 더 자세히 배우게 됩니다!), 그리고 사실 기억(fact memorization)이나 더 기본적인 문법 및 구문 규칙과 같은 더 스마트한 기법들을 사용하기 시작해야 합니다. 이 시점부터는 성능을 조금씩 개선하는 것이 더 어려워지며, 이로 인해 loss curve가 평탄해지게 됩니다.

### 연습 문제 (선택 사항) - completion 로그 기록하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 20-40 minutes on this exercise, if you choose to attempt it.
> Note, you might want to come back to this exercise *after* you learn how sampling works.
> ```

몇 개의 prompt를 선택하여, 해당 문장들에 대한 모델의 completion을 로그로 기록해 보십시오. loss를 기록하는 것보다 더 낮은 빈도로 수행하는 것을 권장합니다 (예: 10-100 batch마다 한 번).

텍스트 로그를 기록하기 위한 `wandb` 문법은 매우 간단합니다. 우선, 단순히 stdout으로 출력을 print하면 Weights & Biases에도 함께 기록됩니다 (실행 결과의 "Logs" 섹션에서 확인할 수 있습니다). 또는, 데이터를 table 형태로 기록하여 다른 차트 옆에 표시되게 할 수도 있습니다:

```python
wandb.log({"completions_table": wandb.Table(
    data = data,
    columns = ["epoch", "step", "text"]
)})
```

여기서 `data`은 (epoch, step, text)를 포함하는 길이 3의 리스트들로 이루어진 리스트입니다. 이 옵션을 선택하는 경우, 너무 많은 데이터가 전송되지 않도록 모델에서 샘플링하는 빈도보다 table을 기록하는 빈도를 더 낮게 설정하는 것을 권장합니다 (안타깝게도 wandb는 로그 기록 중에 table을 점진적으로 업데이트하는 방법을 제공하지 않기 때문입니다).

샘플링 연습 문제(상당히 깁니다!)를 진행하기 전에 이를 시도해보고 싶다면, 아래 코드를 사용하여 모델로부터 출력을 샘플링할 수 있습니다. `TransformerSampler` 객체는 이미 inference 모드이므로 이 부분은 걱정하지 않으셔도 됩니다.

In [ ]:
def sampling_fn(model: DemoTransformer, prompt: str) -> str:
    sampler = solutions.TransformerSampler(model, reference_gpt2.tokenizer)
    output = sampler.sample(prompt, temperature=0.7, top_p=0.95, max_tokens_generated=16)
    return output


model = DemoTransformer(model_cfg).to(device)

# Should be entirely random, because it uses a newly initialized model
print(sampling_fn(model, prompt="John and Mary went to the"))

In [ ]:
# YOUR CODE HERE - rewrite the TransformerTrainer.train method, so that it logs completions


prompt_list = [
    "Eliezer Shlomo Yudkowsky (born September 11, 1979) is an American decision and artificial intelligence (AI) theorist and writer, best known for",
    "In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.",
    "John and Mary went to the",
]

model = DemoTransformer(model_cfg).to(device)
args = TransformerTrainingArgsLogText()
trainer = TransformerTrainer(args, model)
trainer.train(sampling_fn, prompt_list)
# Read full report here - https://api.wandb.ai/links/callum-mcdougall/5ex16e5w

<details><summary>솔루션</summary>

```python
@dataclass
class TransformerTrainingArgsLogText(TransformerTrainingArgs):
    text_sample_freq: int = 20
    table_log_freq: int = 200

    def __post_init__(self):
        assert self.table_log_freq >= self.text_sample_freq, (
            "You should log the table less frequently than you add text to it."
        )


def train_log_text(self: TransformerTrainer, sampling_fn: Callable, prompt_list: list[str]):
    """
    Trains the model, for `self.args.epochs` epochs. Also handles wandb initialisation, and early stopping
    for each epoch at `self.args.max_steps_per_epoch` steps.

    This also takes 2 extra arguments:
        sampling_fn: function which takes model & a single prompt (i.e. text string) and returns text string output
        prompt_list: list of prompts we'll log output on
    """
    wandb.init(project=self.args.wandb_project, name=self.args.wandb_name, config=self.args)
    accuracy = np.nan
    progress_bar = tqdm(total=self.args.max_steps_per_epoch * self.args.epochs)

    # Create a list for storing data
    completions_list = []

    for epoch in range(self.args.epochs):
        for i, batch in enumerate(self.train_loader):
            loss = self.training_step(batch)
            progress_bar.update()
            progress_bar.set_description(f"Epoch {epoch + 1}, loss: {loss:.3f}, accuracy: {accuracy:.3f}")

            # Control the adding of text to the table, and the logging of text
            if self.step % self.args.text_sample_freq == 0:
                text_completions = [sampling_fn(self.model, prompt) for prompt in prompt_list]
                completions_list.append([epoch, self.step, *text_completions])
            if self.step % self.args.table_log_freq == 0:
                wandb.log(
                    {
                        "completions_table": wandb.Table(
                            data=completions_list,
                            columns=[
                                "epoch",
                                "step",
                                *[f"prompt_{i}" for i in range(len(prompt_list))],
                            ],
                        )
                    }
                )

            if i >= self.args.max_steps_per_epoch:
                break

        accuracy = self.evaluate()

    wandb.finish()


TransformerTrainer.train = train_log_text


prompt_list = [
    "Eliezer Shlomo Yudkowsky (born September 11, 1979) is an American decision and artificial intelligence (AI) theorist and writer, best known for",
    "In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English.",
    "John and Mary went to the",
]

model = DemoTransformer(model_cfg).to(device)
args = TransformerTrainingArgsLogText()
trainer = TransformerTrainer(args, model)
trainer.train(sampling_fn, prompt_list)
# Read full report here - https://api.wandb.ai/links/callum-mcdougall/5ex16e5w
```
</details>

모델로부터 완벽한 논리적 일관성을 기대해서는 안 되지만, 적어도 기본적인 단어 빈도를 준수하고 때때로 기본적인 문법 규칙을 따르는 모습은 볼 수 있어야 합니다. 이를 통해 transformer를 학습시키는 것이 얼마나 어려울 수 있는지에 대한 관점을 얻으셨기를 바랍니다!

# 4️⃣ Transformer에서 샘플링하기

> ##### 학습 목표
>
> * transformer에서 샘플링하는 방법을 학습합니다.
>     * 여기에는 greedy search나 top-k와 같은 기본적인 방법부터 beam search와 같은 더 발전된 방법들이 포함됩니다.
> * transformer의 출력을 캐싱하여 텍스트를 더 효율적으로 생성하는 방법을 학습합니다.
>     * 선택 사항으로, 캐싱 방법을 활용하도록 샘플링 함수를 다시 작성해 봅니다.

transformer에서 어떻게 출력을 생성할 수 있을지 논의해 보겠습니다.

분포에서 token을 샘플링하는 가장 분명한 방법 중 하나는 항상 가장 높은 확률이 할당된 token을 선택하는 것입니다. 하지만 이는 지루하고 반복적인 결과로 이어질 수 있으며, 최악의 경우 transformer의 출력이 루프에 빠질 수 있습니다.

먼저, HuggingFace의 블로그 포스트 [How to generate text: using different decoding methods for language generation with Transformers](https://huggingface.co/blog/how-to-generate) 를 읽어보시기 바랍니다. 읽으신 후, 아래의 연습 문제를 시작하시면 됩니다.

## `TransformerSampler` 클래스

아래에 `TransformerSampler` 클래스를 제공합니다. 이 클래스는 다음과 같은 중요한 메서드들을 포함하고 있습니다:

- `sample`: 가장 상위 수준의 메서드입니다. 종료 기준 중 하나가 충족될 때까지 `sample_next_token`을 반복적으로 호출하여 새로운 token들을 생성합니다.
- `sample_next_token`: 일부 하이퍼파라미터를 기반으로 단일 새로운 token을 샘플링합니다. 여기에는 temperature scaling, top-k sampling, top-p sampling 등 다양한 샘플링 방법과 기술이 포함될 수 있습니다.
- 앞서 언급한 샘플링 방법과 기술들을 적용하는 기타 메서드 세트입니다.

`sample_next_token`가 어떻게 작동하는지, 그리고 예시로 greedy sampling이 `greedy_search`을 통해 어떻게 구현되는지 확인할 수 있습니다. 우리는 매 단계에서 가장 높은 logit을 가진 token을 계속해서 선택합니다.

<details>
<summary>질문 - 왜 <code>temperature=0.0</code>이 greedy sampling에 해당한다고 생각하십니까?</summary>

샘플링에 temperature를 적용한다는 것은 (나중에 살펴보겠지만) 모든 logit에 `(1 / temperature)`를 곱해 스케일을 조정하는 것을 의미합니다. 이에 대한 기본적인 직관은 다음과 같습니다:

* temperature가 높을수록 스케일 인자가 작아지므로, 모든 logit이 0에 가까워집니다. 즉, uniform distribution에 가까워지며 샘플링 과정이 훨씬 더 무작위해집니다 (더 다양하고 변화무쌍한 출력을 생성합니다).
* temperature가 낮을수록 스케일 인자가 커지므로, 모든 logit이 무한대에 가까워집니다. 즉, dirac delta function에 가까워지며 샘플링 과정이 훨씬 더 결정론적으로 변합니다 (덜 다양한 출력을 생성합니다).

temperature가 0에 가까워질수록, 가장 큰 logit과 두 번째로 큰 logit 사이의 차이가 매우 커지므로, 분포는 "가장 가능성이 높은 token에 확률 1을 부여"하는 경향을 보입니다. 즉, greedy sampling이 됩니다. 원하신다면 이를 공식적으로 유도해 보실 수 있습니다.
</details>

다음 실습에서는 `sample` 메서드를 구현하고, 이어서 다른 모든 메서드들을 구현하게 됩니다.

### 연습 문제 - `sample` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 25-40 minutes on this exercise.
> ```

`sample` 메서드는 다음과 같은 과정을 반복하여 새로운 token을 autoregressively 생성합니다:

- 현재 token 시퀀스를 모델에 통과시켜 logit을 얻습니다.
- 샘플링 기법을 사용하여 새로운 token을 선택합니다. 즉, `sample_next_token(input_ids, logits, **kwargs)` 을 사용합니다.
- 이 새로운 token을 입력 시퀀스에 추가합니다.
- 다음 종료 조건 중 하나가 충족될 때까지 이 과정을 반복합니다: `max_tokens_generated` 개의 새로운 token을 생성하거나, end-of-sequence token(`self.tokenizer.eos_token_id` 를 통해 접근 가능)을 생성하는 경우입니다.

마지막으로, 샘플링된 문자열을 반환하기 위해 `tokenizer.decode` 메서드를 사용합니다. 또한, 생성되는 동안 디코딩된 시퀀스를 출력하기 위해 `verbose` 인자를 사용할 수 있습니다 (이는 디버깅에 도움이 됩니다).

아래는 greedy sampling(매 단계에서 항상 가장 확률이 높은 다음 token을 선택하는 방식)을 수행하여 작성하신 샘플링 함수를 테스트하는 코드입니다.

몇 가지 힌트입니다:

- tensor shape을 잊지 마십시오! 모델의 입력은 항상 batch 차원을 가져야 합니다. 즉, shape이 `(1, seq_len)` 이어야 합니다.
- `sample_next_token` 메서드는 정수를 반환하므로, 이를 입력 ID의 끝에 연결하기 전에 반드시 tensor로 감싸주십시오.
- 또한 모든 tensor가 동일한 device에 있도록 하십시오 (전역 변수인 `device` 이 있습니다).
- `model.eval()` 을 사용하여 모델을 evaluation mode로 설정하는 것을 기억하십시오.

In [ ]:
class TransformerSampler:
    def __init__(self, model: DemoTransformer, tokenizer: GPT2TokenizerFast):
        self.model = model
        self.cfg = model.cfg
        self.tokenizer = tokenizer

    @t.inference_mode()
    def sample(self, prompt: str, max_tokens_generated=100, verbose=False, **kwargs) -> str:
        """
        Returns a string of autoregressively generated text, starting from the prompt.

        Sampling terminates at max_tokens_generated, or when the model generates an end-of-sequence token. kwargs are
        passed to sample_next_token, to give detailed instructions on how new tokens are chosen.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_next_token(
        input_ids: Int[Tensor, " seq_len"],
        logits: Float[Tensor, "d_vocab"],
        temperature=1.0,
        top_k=0,
        top_p=0.0,
        frequency_penalty=0.0,
        seed=None,
    ) -> int:
        assert input_ids.ndim == 1, "input_ids should be a 1D sequence of token ids"
        assert temperature >= 0, "Temperature should be non-negative"
        assert 0 <= top_p <= 1.0, "Top-p must be a probability"
        assert 0 <= top_k, "Top-k must be non-negative"
        assert not (top_p != 0 and top_k != 0), "At most one of top-p and top-k supported"

        # Set random seeds for reproducibility
        if seed is not None:
            t.manual_seed(seed)
            np.random.seed(seed)

        # Apply all the specialized sampling methods
        if temperature == 0:
            return TransformerSampler.greedy_search(logits)
        elif temperature != 1.0:
            logits = TransformerSampler.apply_temperature(logits, temperature)
        if frequency_penalty != 0.0:
            logits = TransformerSampler.apply_frequency_penalty(input_ids, logits, frequency_penalty)
        if top_k > 0:
            return TransformerSampler.sample_top_k(logits, top_k)
        if top_p > 0.0:
            return TransformerSampler.sample_top_p(logits, top_p)
        return TransformerSampler.sample_basic(logits)

    @staticmethod
    def greedy_search(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Returns the most likely token (as an int).
        """
        raise NotImplementedError()

    @staticmethod
    def apply_temperature(logits: Float[Tensor, "d_vocab"], temperature: float) -> Float[Tensor, "d_vocab"]:
        """
        Applies temperature scaling to the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def apply_frequency_penalty(
        input_ids: Int[Tensor, " seq_len"], logits: Float[Tensor, "d_vocab"], freq_penalty: float
    ) -> Float[Tensor, "d_vocab"]:
        """
        Applies a frequency penalty to the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_basic(logits: Float[Tensor, "d_vocab"]) -> int:
        """
        Samples from the distribution defined by the logits.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_top_k(logits: Float[Tensor, "d_vocab"], k: int) -> int:
        """
        Samples from the top k most likely tokens.
        """
        raise NotImplementedError()

    @staticmethod
    def sample_top_p(logits: Float[Tensor, "d_vocab"], top_p: float, min_tokens_to_keep: int = 1) -> int:
        """
        Samples from the most likely tokens which make up at least p cumulative probability.
        """
        raise NotImplementedError()

    @t.inference_mode()
    def beam_search(
        self,
        prompt: str,
        num_return_sequences: int,
        num_beams: int,
        max_new_tokens: int,
        no_repeat_ngram_size: int | None = None,
    ) -> list[tuple[float, str]]:
        """
        Implements a beam search, by repeatedly performing the `generate` and `filter` steps (starting from the initial
        prompt) until either of the two stopping criteria are met: (1) we've generated `max_new_tokens` tokens, or (2)
        we've generated `num_returns_sequences` terminating sequences.
        """
        raise NotImplementedError()


t.set_grad_enabled(False)  # gradients are not necessary for sampling

model = DemoTransformer(Config()).to(device)
model.load_state_dict(reference_gpt2.state_dict(), strict=False)
tokenizer = reference_gpt2.tokenizer
sampler = TransformerSampler(model, tokenizer)

prompt = "Jingle bells, jingle bells, jingle all the way"
print(f"Testing greedy decoding\nPrompt:   {prompt!r}")

expected = "Jingle bells, jingle bells, jingle all the way up to the top of the mountain."
output = sampler.sample(prompt, max_tokens_generated=8, temperature=0.0)

print(f"Expected: {expected!r}\nActual:   {output!r}\n")
assert output == expected

print("Tests passed!")

<details>
<summary>솔루션</summary>

```python
@t.inference_mode()
def sample(self, prompt: str, max_tokens_generated=100, verbose=False, **kwargs):
    """
    Returns a string of autoregressively generated text, starting from the prompt.

    Sampling terminates at max_tokens_generated, or when the model generates an end-of-sequence token. kwargs are
    passed to sample_next_token, to give detailed instructions on how new tokens are chosen.
    """
    self.model.eval()
    input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)[0]

    for i in range(max_tokens_generated):
        # Get new logits (make sure we don't pass in more tokens than the model's context length)
        logits = self.model(input_ids[None, -self.cfg.n_ctx :])
        # We only take logits for the last token, because this is what we're sampling
        logits = logits[0, -1]
        # Get next token (as a tensor of size (1, 1) so we can concat it to input_ids)
        next_token = t.tensor([TransformerSampler.sample_next_token(input_ids, logits, **kwargs)], device=device)
        # Create new input ids string, with shape (1, old_seq_len + 1)
        input_ids = t.cat([input_ids, next_token], dim=-1)
        # Print out results, if required
        if verbose:
            print(self.tokenizer.decode(input_ids), end="\r")
        # If our new token was the end-of-text token, stop
        if next_token == getattr(self.tokenizer, "eos_token_id", None):
            break

    return self.tokenizer.decode(input_ids)
```

</details>

## Categorical을 이용한 샘플링

이제 구체적인 샘플링 방법들을 구현해 보겠습니다. 각 경우마다 위에서 정의한 클래스로 돌아가 해당 메서드를 채워 넣으시면 됩니다.

PyTorch는 다양한 분포에서 샘플링할 수 있는 여러 편리한 메서드들이 포함된 [`distributions`](https://pytorch.org/docs/stable/distributions.html#distribution) 패키지를 제공합니다.

지금은 [`t.distributions.categorical.Categorical`](https://pytorch.org/docs/stable/distributions.html#categorical)만 필요합니다. 이를 사용하여 제공된 logit(temperature 및 frequency penalty에 의해 이미 수정되었을 수 있음)에서 샘플링하는 `sample_basic`를 구현하십시오.

샘플들을 batch로 처리하지 않기 때문에 속도가 느릴 수 있지만, 지금은 속도에 대해 걱정하지 않으셔도 됩니다.

### 연습 문제 - `sample_basic`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 5-15 minutes on this exercise.
> ```

위의 `TransformerSampler` 클래스에 기본적인 sampling을 구현하고 (즉, `sample_basic` 메서드), 아래 코드를 실행하여 솔루션이 제대로 작동하는지 확인하십시오.

In [ ]:
prompt = "John and Mary went to the"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = model(input_ids)[0, -1]

expected_top_5 = {
    " church": 0.0648,
    " house": 0.0367,
    " temple": 0.0145,
    " same": 0.0104,
    " Church": 0.0097,
}
frequency_of_top_5 = defaultdict(int)

N = 10_000
for _ in tqdm(range(N)):
    token = TransformerSampler.sample_next_token(input_ids.squeeze(), logits)
    frequency_of_top_5[tokenizer.decode(token)] += 1

for word in expected_top_5:
    expected_freq = expected_top_5[word]
    observed_freq = frequency_of_top_5[word] / N
    print(f"Word: {word!r:<9}. Expected freq {expected_freq:.4f}, observed freq {observed_freq:.4f}")
    assert abs(observed_freq - expected_freq) < 0.01, "Try increasing N if this fails by a small amount."

print("Tests passed!")

<details>
<summary>솔루션</summary>

```python
@staticmethod
def sample_basic(logits: Float[Tensor, "d_vocab"]) -> int:
    """
    Samples from the distribution defined by the logits.
    """
    sampled_token = t.distributions.categorical.Categorical(logits=logits).sample()
    return sampled_token.item()
```

</details>

### 연습 문제 - `apply_temperature`

> ```yaml
> Difficulty: 🔴⚪⚪⚪⚪
> Importance: 🔵🔵⚪⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

Temperature는 거창하게 들리지만, 실제로는 단순히 logit을 temperature로 나누는 것입니다. 이제 이를 `TransformerSampler` 클래스에 구현해야 합니다.

In [ ]:
logits = t.tensor([1, 2]).log()

cold_logits = TransformerSampler.apply_temperature(logits, temperature=0.001)
print('A low temperature "sharpens" or "peaks" the distribution: ', cold_logits)
t.testing.assert_close(cold_logits, 1000.0 * logits)

hot_logits = TransformerSampler.apply_temperature(logits, temperature=1000.0)
print("A high temperature flattens the distribution: ", hot_logits)
t.testing.assert_close(hot_logits, 0.001 * logits)

print("Tests passed!")

<details>
<summary>솔루션</summary>

```python
@staticmethod
def apply_temperature(logits: Float[Tensor, "d_vocab"], temperature: float) -> Float[Tensor, "d_vocab"]:
    """
    Applies temperature scaling to the logits.
    """
    return logits / temperature
```

</details>

### 연습 문제 - `apply_frequency_penalty`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 10-15 minutes on this exercise.
> ```

frequency penalty 또한 간단합니다. 각 token의 등장 횟수를 센 다음, 각 등장 횟수마다 `freq_penalty`를 뺍니다. 힌트: 이를 벡터화된 방식으로 수행하려면 `t.bincount`(문서 [here](https://pytorch.org/docs/stable/generated/torch.bincount.html))을 사용하십시오.

이제 ``apply_frequency_penalty`` 메서드를 ``TransformerSampler`` 클래스에 구현한 다음, 아래 셀을 실행하여 솔루션을 확인하시기 바랍니다.

<details>
<summary>도와주세요 - <code>RuntimeError</code>가 발생했습니다. tensor 크기가 일치하지 않습니다.</summary>

`t.bincount`의 문서 페이지를 확인하십시오. `minlength` 인자를 사용해야 할 수도 있습니다 - 그 이유는 무엇일까요?
</details>

In [ ]:
bieber_prompt = "And I was like Baby, baby, baby, oh Like, Baby, baby, baby, no Like, Baby, baby, baby, oh I thought you'd always be mine, mine"
input_ids = tokenizer.encode(bieber_prompt, return_tensors="pt")
logits = t.ones(tokenizer.vocab_size)
penalized_logits = TransformerSampler.apply_frequency_penalty(input_ids.squeeze(), logits, 2.0)

assert penalized_logits[5156].item() == -11, "Expected 6 occurrences of ' baby' with leading space, 1-2*6=-11"
assert penalized_logits[14801].item() == -5, "Expected 3 occurrences of ' Baby' with leading space, 1-2*3=-5"

print("Tests passed!")

<details>
<summary>솔루션</summary>

```python
@staticmethod
def apply_frequency_penalty(
    input_ids: Int[Tensor, "seq_len"], logits: Float[Tensor, "d_vocab"], freq_penalty: float
) -> Float[Tensor, "d_vocab"]:
    """
    Applies a frequency penalty to the logits.
    """
    d_vocab = logits.size(0)
    id_freqs = t.bincount(input_ids, minlength=d_vocab)
    return logits - freq_penalty * id_freqs
```

</details>

### 샘플링 - 수동 테스트

아래 셀을 실행하여 `temperature` 및 `freq_penalty` 인자에 대한 감을 익혀보시기 바랍니다. 직접 프롬프트를 입력하고 다른 값들을 시도해 보십시오.

참고: 모델이 줄바꿈이나 출력되지 않는 문자를 생성할 수 있으므로, 생성된 텍스트에 `print`를 호출하면 화면에 어색하게 보일 때가 있습니다. 출력하기 전에 문자열에 `repr`를 호출하면 문자열이 깔끔하게 이스케이프 처리됩니다.

In [ ]:
sampler = TransformerSampler(model, tokenizer)

N_RUNS = 1
your_prompt = "Jingle bells, jingle bells, jingle all the way"
cases = [
    ("High freq penalty", dict(frequency_penalty=100.0)),
    ("Negative freq penalty", dict(frequency_penalty=-3.0)),
    ("Too hot!", dict(temperature=2.0)),
    ("Pleasantly cool", dict(temperature=0.7)),
    ("Pleasantly warm", dict(temperature=0.9)),
    ("Too cold!", dict(temperature=0.01)),
]

table = Table("Name", "Kwargs", "Output", title="Sampling - Manual Testing")

for name, kwargs in cases:
    for i in range(N_RUNS):
        output = sampler.sample(your_prompt, max_tokens_generated=24, **kwargs)
        table.add_row(name, str(kwargs), repr(output) + "\n")

rprint(table)

## Top-K Sampling

개념적으로, top-k sampling의 단계는 다음과 같습니다:
- 가장 큰 확률 `top_k` 개를 찾습니다 ([`torch.topk`](https://pytorch.org/docs/stable/generated/torch.topk.html) 을 사용할 수 있습니다)
- 나머지 모든 확률을 0으로 설정합니다
- 정규화(Normalize) 후 샘플링합니다

### 연습 문제 - `sample_top_k`

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 5-10 minutes on this exercise.
> ```

이제 `sample_top_k` 메서드를 구현하십시오. 구현 시 전체 과정에서 log-space를 유지해야 합니다 (확률을 얻기 위해 지수 함수를 사용하지 마십시오). 이는 `Categorical`이 정규화되지 않은 logit을 허용하므로, 실제로 정규화에 대해 걱정할 필요가 없음을 의미합니다.

In [ ]:
prompt = "John and Mary went to the"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = model(input_ids)[0, -1]

expected_top_5 = {
    " church": 0.0648,
    " house": 0.0367,
    " temple": 0.0145,
    " same": 0.0104,
    " Church": 0.0097,
}
topk_5_sum = sum(expected_top_5.values())

observed_freqs = defaultdict(int)

N = 10000
for _ in tqdm(range(N)):
    token = TransformerSampler.sample_next_token(input_ids.squeeze(), logits, top_k=5)
    observed_freqs[tokenizer.decode(token)] += 1

for word in expected_top_5:
    expected_freq = expected_top_5[word] / topk_5_sum
    observed_freq = observed_freqs[word] / N
    print(f"Word: {word!r:<9}. Expected freq = {expected_freq:.4f}, observed freq = {observed_freq:.4f}")
    assert abs(observed_freq - expected_freq) < 0.01

<details>
<summary>솔루션</summary>

```python
@staticmethod
def sample_top_k(logits: Float[Tensor, "d_vocab"], k: int) -> int:
    """
    Samples from the top k most likely tokens.
    """
    top_k_logits, top_k_token_ids = logits.topk(k)
    # Get sampled token (which is an index corresponding to the list of top-k tokens)
    sampled_token_idx = t.distributions.categorical.Categorical(logits=top_k_logits).sample()
    # Get the actual token id, as an int
    return top_k_token_ids[sampled_token_idx].item()
```

</details>

[GPT-2 paper](https://d4mucfpksywv.cloudfront.net/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)은 유니콘에 관한 예시 프롬프트를 포함한 것으로 유명합니다. 이제 이 예시가 얼마나 cherry picked 되었는지 직접 확인해 볼 차례입니다.

해당 논문에서는 `top_k=40`과 10개 샘플 중 최적의 결과(best of 10 samples)를 사용했다고 주장합니다.

In [ ]:
sampler = TransformerSampler(model, tokenizer)

your_prompt = "In a shocking finding, scientist discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains. Even more surprising to the researchers was the fact that the unicorns spoke perfect English."

output = sampler.sample(your_prompt, temperature=0.7, top_k=40, max_tokens_generated=64)

rprint(f"Your model said:\n\n[bold dark_orange]{output}")

정말 놀라운 결과입니다! 이와 같은 기본적인 모델들이 얼마나 큰 패러다임의 전환을 가져왔는지 이해하기 위해 [this section from Simulators](https://www.lesswrong.com/posts/vJFdjigzmcXMhNTsx/simulators#The_limit_of_sequence_modeling)를 읽어보시는 것을 추천합니다.

## Top-p 또는 Nucleus Sampling

기본 아이디어는 선택한 단어들의 총 확률이 특정 임계값을 넘을 때까지 가장 가능성이 높은 단어들을 선택하는 것입니다. 그 다음, 선택된 단어들의 logit을 기반으로 샘플링을 수행합니다.

단계는 다음과 같습니다:

- 확률을 큰 순서에서 작은 순서로 정렬합니다.
- 누적 확률이 처음으로 `top_p`와 같거나 이를 초과하는 컷오프 지점을 찾습니다. 임계값을 넘는 첫 번째 확률을 포함하여 컷오프를 수행합니다.
- 유지된 확률의 개수가 `min_tokens_to_keep`보다 적다면, 대신 그만큼의 token을 유지합니다.
- 나머지 모든 확률을 0으로 설정합니다.
- 정규화 후 샘플링합니다.

예를 들어, 확률이 `(0.4, 0.3, 0.2, 0.1)`이고 컷오프가 `top_p=0.8`이라면, 처음 세 개의 요소에서 샘플링하게 됩니다 (이들의 총 확률은 `0.9`로 임계값을 넘지만, 처음 두 개의 총 확률은 `0.7`로 임계값보다 낮기 때문입니다). 이 세 개에서 샘플링하기로 결정했다면, 이들의 합으로 나누어 다시 정규화하며, 따라서 샘플링 시 사용하는 확률은 `(0.4/0.9, 0.3/0.9, 0.2/0.9)`이 됩니다.

선택 사항으로, 다양한 방법론의 비교를 위해 논문 [The Curious Case of Neural Text Degeneration](https://arxiv.org/pdf/1904.09751.pdf)을 참고하시기 바랍니다.

### 연습 문제 - `sample_top_p`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

In [ ]:
prompt = "John and Mary went to the"
input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
logits = model(input_ids)[0, -1]

expected_top_10pct = {
    " church": 0.0648,
    " house": 0.0367,  # These are the two most likely tokens, and add up to >10%
}
top_10pct_sum = sum(expected_top_10pct.values())

observed_freqs = defaultdict(int)

N = 10_000
for _ in tqdm(range(N)):
    token = TransformerSampler.sample_next_token(input_ids.squeeze(), logits, top_p=0.1)
    observed_freqs[tokenizer.decode(token)] += 1

for word in expected_top_10pct:
    expected_freq = expected_top_10pct[word] / top_10pct_sum
    observed_freq = observed_freqs[word] / N
    print(f"Word: {word!r:<9}. Expected freq {expected_freq:.4f}, observed freq {observed_freq:.4f}")
    assert abs(observed_freq - expected_freq) < 0.01, "Try increasing N if this fails by a small amount."

<details>
<summary>도움말 - 이 함수를 어떻게 구현해야 할지 모르겠습니다.</summary>

먼저, `sort(descending=True)` 메서드를 사용하여 logit을 정렬합니다 (이 메서드는 값과 인덱스를 반환합니다). 그런 다음 이 logit에 softmax를 적용하고 cumsum을 구하여 `cumulative_probs`를 얻을 수 있습니다. 이후 `t.searchsorted` 함수를 사용하여 유지할 확률의 개수를 결정할 수 있습니다.

어떤 확률을 유지할지 결정했다면, 원래의 logit을 사용하여 샘플링하는 것이 가장 쉽습니다 (`logits.sort`을 호출했을 때 인덱스를 보존했어야 합니다). 이렇게 하면 확률을 사용할 때처럼 다시 정규화(renormalising)하는 것을 걱정할 필요가 없습니다.
</details>

<details>
<summary>솔루션</summary>

```python
@staticmethod
def sample_top_p(logits: Float[Tensor, "d_vocab"], top_p: float, min_tokens_to_keep: int = 1) -> int:
    """
    Samples from the most likely tokens which make up at least p cumulative probability.
    """
    # Sort logits, and get cumulative probabilities
    logits_sorted, indices = logits.sort(descending=True, stable=True)
    cumul_probs = logits_sorted.softmax(-1).cumsum(-1)
    # Choose which tokens to keep, in the set we sample from
    n_keep = t.searchsorted(cumul_probs, top_p, side="left").item() + 1
    n_keep = max(n_keep, min_tokens_to_keep)
    keep_idx = indices[:n_keep]
    keep_logits = logits[keep_idx]
    # Perform the sampling
    sample = t.distributions.categorical.Categorical(logits=keep_logits).sample()
    return keep_idx[sample].item()
```

</details>

이제 top-p sampling의 예시입니다:

In [ ]:
sampler = TransformerSampler(model, tokenizer)

your_prompt = "Eliezer Shlomo Yudkowsky (born September 11, 1979) is an American decision and artificial intelligence (AI) theorist and writer, best known for"
output = sampler.sample(your_prompt, temperature=0.7, top_p=0.95, max_tokens_generated=64)
rprint(f"Your model said:\n\n[bold dark_orange]{output}")

## Beam search

마지막으로, 출력값을 탐색하는 더 발전된 방법인 **beam search**를 구현하겠습니다. 다음 단계로 넘어가기 전에 beam search에 관한 [HuggingFace page](https://huggingface.co/blog/how-to-generate#beam-search)를 읽어보시기 바랍니다.

beam search에서는 확률의 곱으로 측정했을 때 지금까지 가장 가능성이 높은 `num_beams` 크기의 completion 리스트를 유지합니다. 이 곱은 매우 작아질 수 있으므로, 대신 log 확률의 합을 사용합니다. 주의할 점은 log 확률이 모델의 출력값과 *동일하지 않다*는 것입니다. log 확률은 먼저 출력값에 softmax를 취한 다음 log를 취하여 얻습니다. 이는 [`log_softmax`](https://pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html) 함수 / tensor 메서드로 수행할 수 있습니다.

<details>
<summary>Log 확률은 logit 출력값이 특정 양 X만큼 이동한 것과 같습니다 (여기서 X는 원래 logit 출력값의 함수입니다). 이를 증명할 수 있습니까?</summary>

logit 벡터를 $x$라고 가정하고, softmax를 취해 확률 벡터 $p$를 얻은 다음, 다시 log를 취해 log 확률 벡터 $l$를 얻었다고 합시다. 그러면 이 logprobs 벡터의 $i$번째 요소는 다음과 같습니다:

$$
\begin{align}
l_i &= \log p_i \\
&= \log \frac{\exp(x_i)}{\sum_j \exp(x_j)} \\
&= x_i - \log \sum_j \exp(x_j) \\
&= x_i - C
\end{align}
$$

여기서 $C = \log \sum_j \exp(x_j)$은 모든 요소에 대해 동일합니다. 따라서 $l_i$는 logit 출력값 $x_i$가 $C$만큼 이동한 것과 같음을 알 수 있습니다.

logit과 logprob를 혼동하지 않는 것이 중요합니다!
</details>

<details>
<summary>왜 logit 출력 대신 log softmax를 사용한다고 생각하십니까?</summary>

logit 출력은 translation invariant(평행 이동 불변)합니다. 만약 두 개의 서로 다른 beam이 있고 그 beam들에서 다음 token들을 생성하고 있다면, 분포를 변경하지 않고도 한 beam의 logit 벡터를 상수만큼 이동시킬 수 있기 때문에 두 beam을 서로 비교할 합리적인 방법이 없을 것입니다.

</details>

매 반복마다 completion 배치를 모델에 통과시키고 log-softmax를 취하여 각 completion에 대해 `d_vocab`개의 log-probs를 얻으며, 총 `num_beams * d_vocab`개의 가능한 다음 completion을 얻습니다.

만약 이 모든 것을 유지한다면 다음 반복 후에 `num_beams * d_vocab * d_vocab`개의 completion을 갖게 되는데, 이는 너무 많습니다. 따라서 대신 점수순으로 정렬하고 가장 좋은(가장 높은) log 확률부터 가장 나쁜(가장 낮은) 확률까지 루프를 돕니다.

아래의 그림이 도움이 될 것입니다 (이 방법의 실제 결과에 기반함). 여기에서 다음과 같은 하이퍼파라미터를 사용합니다:

```python
num_beams = 3
max_new_tokens = 3
num_return_sequences = 2
```

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/beam-search-3.png" width="1000">

각 "generate" 단계 이후에 `num_beams ** 2`개의 가능한 completion이 생성되고, 이를 다시 `num_beams`개로 필터링하는 방식에 주목하십시오. 이는 전체적으로 가장 좋은 `num_beams`개의 completion을 찾기 위해 이만큼의 수가 필요하기 때문입니다. 예를 들어, 길이 `n+1`의 가장 좋은 모든 beam이 길이 `n`의 동일한 beam에서 나왔을 가능성이 있으며, 이 경우 해당 단일 beam에서 생성된 `num_beams`개를 모두 유지해야 합니다.

조기에 종료되는 시퀀스(즉, EOS token을 생성하는 경우)는 어떻게 처리할까요? 정답은 이를 마지막에 반환할 completion 리스트에 추가하고, 생성 트리에서는 제거하는 것입니다. 우리의 알고리즘은 모든 시퀀스의 길이가 초기 prompt 길이보다 `max_new_tokens`만큼 커지거나, `num_returns_sequences`개의 종료 시퀀스를 생성했을 때 종료됩니다.

### 연습 문제 - `beam_search` 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵⚪⚪⚪⚪
> 
> You should spend up to 30-50 minutes on this exercise.
> ```

아래에 `beam_search`의 구현체 하나를 제공해 드렸으며, 이는 `Beams` 클래스의 `generate` 및 `filter` 메서드를 호출합니다 (이는 위 다이어그램의 두 단계에 해당합니다). `beam_search` 메서드는 다음과 같이 작동합니다:

- 최종 출력을 저장하기 위해 (logprob 합계, 문자열 완성본) 튜플 형태의 리스트 `final_logprobs_and_completions`를 생성합니다.
- 생성(새로운 beam 세트 생성)과 필터링(이러한 조합에서 최적의 beam 선택) 단계를 `max_new_tokens`번 수행하며, 동시에 종료된 beam들을 최적 beam 리스트에 추가합니다.
- 종료된 beam들과 단계 종료 시점에 가지고 있는 최적의 beam들을 반환합니다.

따라서 여러분이 해야 할 일은 `generate` 및 `filter` 메서드를 채우는 것입니다. 아래에서 `generate` 및 `filter` 메서드에 대한 몇 가지 유닛 테스트를 확인하실 수 있습니다. 이 테스트들을 통과하면 전체 `beam_search` 함수를 실행할 수 있습니다.

**중요 참고 사항** - 기본적으로 beam search는 많은 반복 단어 / 구 / 문장을 생성합니다. 이는 타당한 결과입니다. 만약 모델이 beam search 공간의 대부분의 완성본보다 훨씬 높은 logit 합계를 가진 완성본을 발견한다면, 문맥상 큰 의미가 없더라도 이 완성본을 반복하고 싶어 할 것입니다. 일반적인 해결책은 n-gram의 반복을 금지하는 것이며, 여러분도 아래 함수에 이를 구현해야 합니다. 다시 말해, `generate` 메서드에서 `logprobs.topk(k)`를 취해 각 시퀀스에서 token을 샘플링하는 대신, 길이가 `no_repeat_ngram_size`인 반복 n-gram을 생성하는 token들을 필터링한 후 상위 `k`개 token을 취해야 합니다. 이 파라미터로 시도해 볼 만한 좋은 값은 2 또는 3입니다 (하지만 먼저 이 파라미터 없이 시도하여 얼마나 차이가 나는지 확인하는 것을 권장합니다!).

In [ ]:
@dataclass
class Beams:
    """Class to store beams during beam search."""

    model: DemoTransformer
    tokenizer: GPT2TokenizerFast
    logprob_sums: Float[Tensor, " batch"]
    tokens: Int[Tensor, "batch seq"]

    def __getitem__(self, batch_idx) -> "Beams":
        """Allows you to create new beams from old beams by slicing along batch dim (useful for `filter`)."""
        return Beams(self.model, self.tokenizer, self.logprob_sums[batch_idx], self.tokens[batch_idx])

    @property
    def logprobs_and_completions(self) -> list[tuple[float, str]]:
        """Returns self as a list of logprob sums and completions (useful for getting final output)."""
        return [
            (logprob_sum.item(), self.tokenizer.decode(tokens))
            for (logprob_sum, tokens) in zip(self.logprob_sums, self.tokens)
        ]

    def generate(self, k: int, no_repeat_ngram_size: int | None = None) -> "Beams":
        """
        Starting from the current set of beams (i.e. self.tokens) and returns a new set of `len(self.tokens) * k` beams,
        containing the best `k` continuations for each of the original beams.

        Optional argument `no_repeat_ngram_size` means your model won't generate any sequences with a repeating n-gram
        of this length.
        """
        raise NotImplementedError()

    def filter(self, k: int) -> tuple["Beams", "Beams"]:
        """
        Returns:
            best_beams: Beams
                filtered version of self, containing all best `k` which are also not terminated.
            early_terminations: Beams
                filtered version of self, containing all best `k` which are also terminated.
        """
        raise NotImplementedError()


    def print(self, title="Best completions", max_print_chars=80) -> None:
        """
        Prints out a set of sequences with their corresponding logprob sums.
        """
        if len(self.tokens) == 0:
            return
        table = Table("logprob sum", "completion", title=title)
        for logprob_sum, tokens in zip(self.logprob_sums, self.tokens):
            text = self.tokenizer.decode(tokens)
            if len(repr(text)) > max_print_chars:
                text = text[: int(0.3 * max_print_chars)] + " ... " + text[-int(0.7 * max_print_chars) :]
            table.add_row(f"{logprob_sum:>8.3f}", repr(text))
        rprint(table)


@t.inference_mode()
def beam_search(
    self: TransformerSampler,
    prompt: str,
    num_return_sequences: int,
    num_beams: int,
    max_new_tokens: int,
    no_repeat_ngram_size: int | None = None,
) -> list[tuple[float, str]]:
    """
    Implements a beam search, by repeatedly performing the `generate` and `filter` steps (starting from the initial
    prompt) until either of the two stopping criteria are met: (1) we've generated `max_new_tokens` tokens, or (2)
    we've generated `num_returns_sequences` terminating sequences.
    """
    assert num_return_sequences <= num_beams
    self.model.eval()

    tokens = self.tokenizer.encode(prompt, return_tensors="pt").to(device)

    final_logprobs_and_completions = []  # we add to this list as we get terminated beams
    best_beams = Beams(self.model, self.tokenizer, t.tensor([0.0]).to(device), tokens)  # start with just 1 beam

    for _ in tqdm(range(max_new_tokens)):
        t.cuda.empty_cache()

        # Generate & filter beams
        best_beams = best_beams.generate(k=num_beams, no_repeat_ngram_size=no_repeat_ngram_size)
        best_beams, best_beams_terminated = best_beams.filter(k=num_beams)

        # Add terminated beams to our list, and return early if we have enough
        final_logprobs_and_completions.extend(best_beams_terminated.logprobs_and_completions)
        if len(final_logprobs_and_completions) >= num_return_sequences:
            return final_logprobs_and_completions[:num_return_sequences]

    # Return terminated beams plus the best ongoing beams of length `orig_len + max_new_tokens`
    final_logprobs_and_completions.extend(best_beams.logprobs_and_completions)
    return final_logprobs_and_completions[:num_return_sequences]


TransformerSampler.beam_search = beam_search

<details>
<summary>도움이 필요합니다 - <code>no_repeat_ngram_size</code> 구현 중에 막혔습니다.</summary>

다음은 `logprobs.topk(k)` 대신 `generate` 함수에서 사용할 수 있는 방법으로, `self.tokens`에 이미 나타난 길이 `no_repeat_ngram_size`의 ngrams를 필터링합니다:

```python
def get_topk_non_repeating(
    self,
    logprobs: Float[Tensor, "batch d_vocab"],
    no_repeat_ngram_size: int | None,
    k: int,
) -> tuple[Float[Tensor, "k"], Int[Tensor, "k"]]:
    """
    logprobs:
        tensor of the log-probs for the next token
    no_repeat_ngram_size:
        size of ngram to avoid repeating
    k:
        number of top logits to return, for each beam in our collection

    Returns:
        equivalent to the output of `logprobs.topk(dim=-1)`, but makes sure that no returned tokens would produce an
        ngram of size `no_repeat_ngram_size` which has already appeared in `self.tokens`.
    """
    batch, seq_len = self.tokens.shape

    # If completion isn't long enough for a repetition, or we have no restrictions, just return topk
    if (no_repeat_ngram_size is not None) and (seq_len > no_repeat_ngram_size - 1):
        # Otherwise, we need to check for ngram repetitions
        # First, get the most recent `no_repeat_ngram_size-1` tokens
        last_ngram_prefix = self.tokens[:, seq_len - (no_repeat_ngram_size - 1) :]
        # Next, find all the tokens we're not allowed to generate, by checking all past ngrams for a match
        for i in range(seq_len - (no_repeat_ngram_size - 1)):
            ngrams = self.tokens[:, i : i + no_repeat_ngram_size]  # (batch, ngram)
            ngrams_are_repeated = (ngrams[:, :-1] == last_ngram_prefix).all(-1)  # (batch,)
            ngram_end_tokens = ngrams[:, [-1]]  # (batch, 1)
            # Fill logprobs with neginf wherever the ngrams are repeated
            logprobs[range(batch), ngram_end_tokens] = t.where(
                ngrams_are_repeated, -1.0e4, logprobs[range(batch), ngram_end_tokens]
            )

    # Finally, get our actual tokens
    return logprobs.topk(k=k, dim=-1)
```

</details>

<details>
<summary>솔루션</summary>

```python
def generate(self, k: int, no_repeat_ngram_size: int | None = None) -> "Beams":
    """
    Starting from the current set of beams (i.e. self.tokens) and returns a new set of `len(self.tokens) * k` beams,
    containing the best `k` continuations for each of the original beams.

    Optional argument `no_repeat_ngram_size` means your model won't generate any sequences with a repeating n-gram
    of this length.
    """
    # Get the output logprobs for the next token (for every sequence in current beams)
    logprobs = self.model(self.tokens)[:, -1, :].log_softmax(-1)

    # Get the top `toks_per_beam` tokens for each sequence
    topk_logprobs, topk_tokenIDs = self.get_topk_non_repeating(logprobs, no_repeat_ngram_size, k=k)

    # Add new logprobs & concat new tokens. When doing this, we need to add an extra `k` dimension since our current
    # logprobs & tokens have shape (batch,) and (batch, seq), but our new ones both have shape (batch, k)
    new_logprob_sums = einops.repeat(self.logprob_sums, "b -> b k", k=k) + topk_logprobs
    new_tokens = t.concat([einops.repeat(self.tokens, "b s -> b k s", k=k), topk_tokenIDs.unsqueeze(-1)], dim=-1)

    return Beams(self.model, self.tokenizer, new_logprob_sums.flatten(), new_tokens.flatten(0, 1))

def filter(self, k: int) -> tuple["Beams", "Beams"]:
    """
    Returns:
        best_beams: Beams
            filtered version of self, containing all best `k` which are also not terminated.
        early_terminations: Beams
            filtered version of self, containing all best `k` which are also terminated.
    """
    # Get the indices of top `k` beams
    top_beam_indices = self.logprob_sums.topk(k=k, dim=0).indices.tolist()
    # Get the indices of terminated sequences
    new_tokens = self.tokens[:, -1]
    terminated_indices = t.nonzero(new_tokens == self.tokenizer.eos_token_id)

    # Get the indices of the `k` best sequences (some terminated, some not terminated)
    best_continuing = [i for i in top_beam_indices if i not in terminated_indices]
    best_terminated = [i for i in top_beam_indices if i in terminated_indices]

    # Return the beam objects from these indices
    return self[best_continuing], self[best_terminated]

def get_topk_non_repeating(
    self,
    logprobs: Float[Tensor, "batch d_vocab"],
    no_repeat_ngram_size: int | None,
    k: int,
) -> tuple[Float[Tensor, "k"], Int[Tensor, "k"]]:
    """
    logprobs:
        tensor of the log-probs for the next token
    no_repeat_ngram_size:
        size of ngram to avoid repeating
    k:
        number of top logits to return, for each beam in our collection

    Returns:
        equivalent to the output of `logprobs.topk(dim=-1)`, but makes sure that no returned tokens would produce an
        ngram of size `no_repeat_ngram_size` which has already appeared in `self.tokens`.
    """
    batch, seq_len = self.tokens.shape

    # If completion isn't long enough for a repetition, or we have no restrictions, just return topk
    if (no_repeat_ngram_size is not None) and (seq_len > no_repeat_ngram_size - 1):
        # Otherwise, we need to check for ngram repetitions
        # First, get the most recent `no_repeat_ngram_size-1` tokens
        last_ngram_prefix = self.tokens[:, seq_len - (no_repeat_ngram_size - 1) :]
        # Next, find all the tokens we're not allowed to generate, by checking all past ngrams for a match
        for i in range(seq_len - (no_repeat_ngram_size - 1)):
            ngrams = self.tokens[:, i : i + no_repeat_ngram_size]  # (batch, ngram)
            ngrams_are_repeated = (ngrams[:, :-1] == last_ngram_prefix).all(-1)  # (batch,)
            ngram_end_tokens = ngrams[:, [-1]]  # (batch, 1)
            # Fill logprobs with neginf wherever the ngrams are repeated
            logprobs[range(batch), ngram_end_tokens] = t.where(
                ngrams_are_repeated, -1.0e4, logprobs[range(batch), ngram_end_tokens]
            )

    # Finally, get our actual tokens
    return logprobs.topk(k=k, dim=-1)
```

</details>

위 다이어그램에 해당하는 `Beams` 클래스와 `print` 메서드의 사용 예시입니다:

In [ ]:
# Start with prompt "When I was", get top 3 tokens (and their logprobs), and use that to create & display the top 3 beams
prompt = "When I was"
tokens = tokenizer.encode(prompt, return_tensors="pt").to(device)
logprobs = model(tokens)[0, -1].log_softmax(-1)
top_logprobs, top_tokens = logprobs.topk(k=3, dim=-1)

new_tokens = t.concat([tokens.repeat(3, 1), top_tokens.unsqueeze(-1)], dim=-1)

beams = Beams(model, tokenizer, logprob_sums=top_logprobs, tokens=new_tokens)
beams.print()

그리고 여기 `"When I was"` 프롬프트부터 시작하여, 여러분의 `generate` 및 `filter` 메서드를 위한 몇 가지 unit test가 있습니다 (따라서 여러분의 출력은 위의 다이어그램과 일치해야 합니다).

In [ ]:
print("Testing generate...")
new_beams = beams.generate(k=3, no_repeat_ngram_size=1)
new_beams.print()

expected_values = [
    (-3.1, "When I was a kid"),
    (-4.8, "When I was a child"),
    (-4.9, "When I was a little"),
]

for i, (logprob_sum, completion) in enumerate(new_beams.logprobs_and_completions[:3]):
    assert abs(logprob_sum - expected_values[i][0]) < 0.1, f"{i}"
    assert completion == expected_values[i][1], f"{i}"

print("All tests for `generate` passed!")

In [ ]:
print("Testing `filter`...")

best_beams, terminated_beams = new_beams.filter(3)
best_beams.print()

expected_values = [
    (-3.1, "When I was a kid"),
    (-3.2, "When I was growing up"),
    (-4.6, "When I was in the"),
]

for i, (logprob_sum, completion) in enumerate(best_beams.logprobs_and_completions):
    assert abs(logprob_sum - expected_values[i][0]) < 0.1, f"{i}"
    assert completion == expected_values[i][1], f"{i}"

assert len(terminated_beams.logprobs_and_completions) == 0

print("All tests for `filter` passed!")

마지막으로, `no_repeat_ngram_size` 인자를 테스트하겠습니다. 이를 위해 시작 beam `beams` 으로부터 새로운 token들을 계속해서 생성하고, 모델이 `I was` ngram을 반복하는지 확인합니다 (n-gram 반복을 금지하지 않는 한 기본적으로 반복하게 됩니다).

In [ ]:
print("Testing `no_repeat_ngram_size`...")

new_beams = beams
for _ in range(5):
    new_beams = new_beams.generate(k=1)
new_beams.print(title="Completions with no ngram restriction")
assert all("I was" in completion.removeprefix(prompt) for _, completion in new_beams.logprobs_and_completions), (
    "Without restriction, all beams should be completed as '...I was...'"
)

new_beams = beams
for _ in range(5):
    new_beams = new_beams.generate(k=1, no_repeat_ngram_size=2)
new_beams.print(title="Completions with no repeated bigrams")
assert all("I was" not in completion.removeprefix(prompt) for _, completion in new_beams.logprobs_and_completions), (
    "With no repeated bigrams, no beams should contain a second '...I was...'"
)

이 모든 unit test를 통과했다면, 전체 beam search 함수 구현을 시도해 볼 수 있습니다. 이 함수는 초기 prompt로부터 `Beams` 객체를 생성하고, 중단 기준이 충족될 때까지 `generate`과 `filter`를 반복적으로 호출해야 합니다.

In [ ]:
sampler = TransformerSampler(model, tokenizer)

prompt = "The ships hung in the sky in much the same way that"
orig_len = len(tokenizer.encode(prompt))

final_logitsums_and_completions = sampler.beam_search(
    prompt=prompt,
    num_return_sequences=3,
    num_beams=40,
    max_new_tokens=60,
    no_repeat_ngram_size=2,
)

# Print all the best output
for logprob_sum, text in final_logitsums_and_completions:
    avg_logprob_as_prob = t.tensor(logprob_sum / (len(tokenizer.encode(text)) - orig_len)).exp()
    rprint(f"Avg token prob = {avg_logprob_as_prob:.3f}\nBest output:\n[bold dark_orange]{text}")

## KV Caching

*이 섹션 또한 도전적으로 설계되었으며, 시간이 꽤 소요될 것입니다. 이를 해결하는 방법은 매우 다양하며, 여러분이 직접 자신만의 방법을 찾아내기를 기대합니다 (드롭다운의 제안을 확인하기 전에 잠시 동안 스스로 고민해 보시기 바랍니다). 또한, 다른 섹션들에 비해 흥미가 덜할 수도 있습니다. 이 경우, 시간적 여유가 많다면 이 장의 "building BERT" 연습 문제를 먼저 시작하셔도 좋습니다.*

### 캐싱이 어떻게 도움이 될 수 있을까요?

지금까지 수행한 텍스트 생성은 특정 값들을 불필요하게 재계산하고 있으며, 이는 더 긴 시퀀스를 생성하려고 할 때 매우 두드러지게 나타납니다.

텍스트를 생성하고 있다고 가정해 봅시다. 이미 "My life motto:"라는 문장에 대해 GPT를 실행했습니다. 이제 "My life motto: Always"라는 문장에 대해 모델을 실행하려고 합니다. 첫 번째 문장에서 계산한 내용 중 어떤 것을 재사용할 수 있을까요?

<details>
<summary>정답</summary>

각 attention 레이어에서, attention 레이어가 이전 시퀀스 위치로부터 필요로 하는 유일한 것은 key와 value 벡터입니다. 이는 캐싱을 사용했을 때와 사용하지 않았을 때의 attention 레이어를 비교한 다음 다이어그램에서 설명됩니다 (다이어그램이 크므로 별도의 창에서 열어 확대해서 보시는 것을 권장합니다).


<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/tl-cache-full.png" width="1200">

</details>

### 연습 문제 - KV caching 구현하기

> ```yaml
> Difficulty: 🔴🔴🔴🔴🔴
> Importance: 🔵⚪⚪⚪⚪
> 
> You are expected to spend well over an hour on this exercise, if you choose to do it.
> ```

GPT-2가 선택적으로 cache를 사용할 수 있도록 수정하십시오. `"My life motto:"`을 입력으로 GPT를 실행할 때, 필요한 값들을 cache에 저장해야 합니다. 그 후 `" Always"`만을 입력으로 하는 다음 forward pass에서는, 값들을 다시 계산하는 대신 cache된 값들을 로드하고 (그리고 cache를 업데이트) 해야 합니다. 이는 단일 입력 시퀀스(batch size 1)에 대해서만 작동하면 되며, 첫 번째 forward pass 이후의 입력은 단 하나의 token이라고 가정해도 좋습니다.

cache의 설계는 전적으로 여러분의 자유입니다. 코드를 작성하기 전에 파트너와 함께 가능한 설계 방안을 논의하십시오. 하나의 GPT2 인스턴스와 동시에 여러 개의 서로 다른 cache 인스턴스를 가질 수 있어야 합니다. [AI Dungeon](https://aidungeon.io/)에서와 같이 텍스트 생성을 요청하는 여러 사용자에게 서비스를 제공하기 위해 하나의 인스턴스를 사용하는 상황을 상상해 보십시오.

또한, 이 기능을 작동시키기 위해 `DemoTransformer` 코드의 일부를 다시 작성해야 합니다. 테스트 케이스들은 모듈이 출력만을 반환하는 대신 튜플의 첫 번째 요소로 출력을 반환하는 경우(즉, `(output, cache)`)를 수용하도록 설계되었으므로, 테스트를 통해 모듈이 여전히 예상대로 작동하는지 확인하십시오.

몇 가지 고려 사항 예시입니다:

* 어떤 GPT-2 클래스들이 cache와 상호작용해야 합니까?
    * positional embedding을 변경해야 합니까? 만약 그렇다면 어떻게 변경해야 합니까?
* cache가 mutable하여 제자리에서(in place) 업데이트되어야 합니까, 아니면 업데이트 시 실제로 별도의 인스턴스를 생성해야 합니까?
    * *(힌트 - beam search 중에 cache를 어떻게 사용할지 생각해 보십시오.)*
* 다른 프로그래머가 여러분의 cache를 잘못 사용할 가능성이 있습니까? 이러한 실패 모드를 방지하거나, 최소한 이를 감지하고 강력하게 경고할 방법이 있습니까?

<details>
<summary>Cache 구현 (예시)</summary>

이 KeyValueCache 객체는 단순히 정교한 tensor로 구성되어 있습니다 (Tensor의 모든 메서드를 상속받습니다). 주요 차이점은 Config 객체로부터 빈 cache를 생성하는 것과 같은 몇 가지 추가 헬퍼 메서드를 가지고 있다는 점입니다.

이를 구현하는 다른 방법들도 있습니다. 예를 들어, `KeyValueCache` 클래스가 `KeyValueCacheEntry` 객체들의 리스트를 포함하게 하는 방식이 있습니다 (여기서 각 객체는 서로 다른 layer에 대응합니다).

```python
# Define a type for a single layer's cache entry (useful for type checking in later functions)
KeyValueCacheTensor = Float[Tensor, "2 batch seq_len n_heads d_head"]

class KeyValueCache(Tensor):
    '''
    This class holds tensors of key and value vectors, to be used for caching.

    If we define it using cfg and batch then it's initialized as empty, but
    we can also define it from kv_cache_entries.
    '''
    @classmethod
    def new_empty(cls, cfg: Config, batch: int = 1) -> "KeyValueCache":
        '''
        Doing a forward pass on a cache created in this way indicates "we don't
        yet have a cache, but we want this forward pass to return a cache".
        Whereas using cache=None in a forward pass indicates we don't want to
        return a cache.
        '''
        shape = (cfg.n_layers, 2, batch, 0, cfg.n_heads, cfg.d_head)
        return cls(*shape).to(device)

    # Define a handful of properties, so they can be referenced directly rather than
    # indexing (which is more likely to lead to mistakes)

    @property
    def k(self) -> Tensor:
        return self[:, 0]

    @property
    def v(self) -> Tensor:
        return self[:, 1]

    @property
    def batch(self) -> int:
        return self.shape[2]

    @property
    def seq_len(self) -> int:
        return self.shape[3]


# Example implementation:
cfg = model.cfg
batch = 6
kv_cache = KeyValueCache.new_empty(cfg, batch)

print(f"Shape of all kv-cache = {tuple(kv_cache.shape)}")
print(f"Shape of just k-cache = {tuple(kv_cache.k.shape)}")
for kv_cache_entry in kv_cache:
    print(f"Shape of cache entry for one layer = {tuple(kv_cache_entry.shape)}")
    break
print(f"Batch size = {kv_cache.batch}")
print(f"Current sequence length = {kv_cache.seq_len}")
```

</details>

<details>
<summary>새로운 <code>DemoTransformer</code> 컴포넌트 (및 테스트)</summary>

```python
# Define new model parts where necessary, and create a new model & test it
# Note that sometimes our modules return a tuple of (tensor output, cache) rather than just output. The
# tests have been built to accommodate this.


class PosEmbed(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_pos = nn.Parameter(t.empty((cfg.n_ctx, cfg.d_model)))
        nn.init.normal_(self.W_pos, std=self.cfg.init_range)

    def forward(
        self,
        tokens: Int[Tensor, "batch position"],
        past_kv_pos_offset: int = 0
    ) -> Float[Tensor, "batch position d_model"]:

        batch, seq_len = tokens.shape
        return einops.repeat(
            self.W_pos[past_kv_pos_offset: seq_len+past_kv_pos_offset],
            "seq d_model -> batch seq d_model",
            batch=batch
        )


class Attention(nn.Module):
    IGNORE: Float[Tensor, ""]

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.W_Q = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_K = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_V = nn.Parameter(t.empty((cfg.n_heads, cfg.d_model, cfg.d_head)))
        self.W_O = nn.Parameter(t.empty((cfg.n_heads, cfg.d_head, cfg.d_model)))
        self.b_Q = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_K = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_V = nn.Parameter(t.zeros((cfg.n_heads, cfg.d_head)))
        self.b_O = nn.Parameter(t.zeros((cfg.d_model)))
        nn.init.normal_(self.W_Q, std=self.cfg.init_range)
        nn.init.normal_(self.W_K, std=self.cfg.init_range)
        nn.init.normal_(self.W_V, std=self.cfg.init_range)
        nn.init.normal_(self.W_O, std=self.cfg.init_range)
        self.register_buffer("IGNORE", t.tensor(-1e5, dtype=t.float32, device=device))

    def forward(
        self,
        normalized_resid_pre: Float[Tensor, "batch posn d_model"],
        kv_cache_entry: KeyValueCacheTensor | None = None,
    ) -> tuple[
        Float[Tensor, "batch posn d_model"],
        KeyValueCacheTensor | None
    ]:
        '''
        Returns the result of applying attention layer to normlized_resid_pre, as well as
        the new cached key and value vectors (which we get from concatenating the old cached
        ones with the new key and value vectors).
        '''
        # Calculate the new query, key and value vectors
        q = einops.einsum(
            normalized_resid_pre, self.W_Q,
            "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head"
        ) + self.b_Q
        k = einops.einsum(
            normalized_resid_pre, self.W_K,
            "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head"
        ) + self.b_K
        v = einops.einsum(
            normalized_resid_pre, self.W_V,
            "batch posn d_model, nheads d_model d_head -> batch posn nheads d_head"
        ) + self.b_V

        # If cache_entry is not None, this means we use the previous key and value vectors
        # Also we'll need to get a new cache entry which will be used later to construct a new cache
        if kv_cache_entry is not None:
            k = t.concat([kv_cache_entry[0], k], dim=1)
            v = t.concat([kv_cache_entry[1], v], dim=1)
            kv_cache_entry = t.stack([k, v])

        # Calculate attention scores, then scale and mask, and apply softmax to get probabilities
        attn_scores = einops.einsum(
            q, k,
            "batch posn_Q nheads d_head, batch posn_K nheads d_head -> batch nheads posn_Q posn_K"
        )
        attn_scores_masked = self.apply_causal_mask(attn_scores / self.cfg.d_head ** 0.5)
        attn_pattern = attn_scores_masked.softmax(-1)

        # Take weighted sum of value vectors, according to attention probabilities
        z = einops.einsum(
            v, attn_pattern,
            "batch posn_K nheads d_head, batch nheads posn_Q posn_K -> batch posn_Q nheads d_head"
        )

        # Calculate output (by applying matrix W_O and summing over heads, then adding bias b_O)
        out = einops.einsum(
            z, self.W_O,
            "batch posn_Q nheads d_head, nheads d_head d_model -> batch posn_Q d_model"
        ) + self.b_O

        return out, kv_cache_entry

    def apply_causal_mask(
        self, attn_scores: Float[Tensor, "batch n_heads query_pos key_pos"]
    ) -> Float[Tensor, "batch n_heads query_pos key_pos"]:
        '''
        Here, attn_scores have shape (batch, n_heads, query_pos, key_pos), where query_pos represents the
        new (non-cached) positions, and key_pos represent all the positions (cached and non-cached).

        So when we create our mask, the query indices and key indices will both go up to the same value
        (the full sequence length), but the query indices will start at >0.
        '''
        new_seq_len, full_seq_len = attn_scores.shape[-2:]
        assert new_seq_len <= full_seq_len
        q_posn = einops.repeat(attn_scores.new_tensor(range(full_seq_len-new_seq_len, full_seq_len)), "q -> q k", k=full_seq_len)
        k_posn = einops.repeat(attn_scores.new_tensor(range(full_seq_len)), "k -> q k", q=new_seq_len)
        mask = q_posn < k_posn
        attn_scores = attn_scores.masked_fill(mask, self.IGNORE)
        return attn_scores


class TransformerBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.ln1 = LayerNorm(cfg)
        self.attn = Attention(cfg)
        self.ln2 = LayerNorm(cfg)
        self.mlp = MLP(cfg)

    def forward(
        self,
        resid_pre: Float[Tensor, "batch position d_model"],
        kv_cache_entry: KeyValueCacheTensor | None = None,
    ) -> Float[Tensor, "batch position d_model"]:

        attn_out, kv_cache_entry = self.attn(self.ln1(resid_pre), kv_cache_entry)
        resid_mid = attn_out + resid_pre
        resid_post = self.mlp(self.ln2(resid_mid)) + resid_mid
        return resid_post, kv_cache_entry


class DemoTransformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed = Embed(cfg)
        self.pos_embed = PosEmbed(cfg)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final = LayerNorm(cfg)
        self.unembed = Unembed(cfg)

    def forward(
        self,
        tokens: Int[Tensor, "batch seq_pos"],
        kv_cache: KeyValueCache | None = None
    ) -> Float[Tensor, "batch position d_vocab"]:

        using_kv_cache = kv_cache is not None

        if using_kv_cache:
            # If using kv_cache, then we only need to pass forward the newest tokens
            # Remember to add positional offset!
            n_cached_tokens = kv_cache.seq_len
            tokens = tokens[:, n_cached_tokens:]
            residual = self.embed(tokens) + self.pos_embed(tokens, n_cached_tokens)
        else:
            # If not using cache, turn it into a list of None's (so we can iterate through it)
            kv_cache = [None for _ in range(self.cfg.n_layers)]
            residual = self.embed(tokens) + self.pos_embed(tokens)

        # Apply all layers, and create a (new) kv_cache from the key & value vectors
        new_kv_cache_entries: list[KeyValueCacheTensor] = []
        for block, kv_cache_entry in zip(self.blocks, kv_cache):
            residual, kv_cache_entry = block(residual, kv_cache_entry)
            if using_kv_cache: new_kv_cache_entries.append(kv_cache_entry)

        logits = self.unembed(self.ln_final(residual))

        if using_kv_cache:
            return logits, KeyValueCache(t.stack(new_kv_cache_entries))
        else:
            return logits, None


tokens = reference_gpt2.to_tokens(reference_text).to(device)
logits, cache = reference_gpt2.run_with_cache(tokens)

rand_int_test(PosEmbed, [2, 4])
load_gpt2_test(PosEmbed, reference_gpt2.pos_embed, tokens)
rand_float_test(Attention, [2, 4, 768])
load_gpt2_test(Attention, reference_gpt2.blocks[0].attn, cache["normalized", 0, "ln1"])
rand_float_test(TransformerBlock, [2, 4, 768])
load_gpt2_test(TransformerBlock, reference_gpt2.blocks[0], cache["resid_pre", 0])
rand_int_test(DemoTransformer, [2, 4])
load_gpt2_test(DemoTransformer, reference_gpt2, tokens)
```

</details>

<details>
<summary>새로운 sampling 함수</summary>

```python
@t.inference_mode()
def sample_with_cache(
    self: TransformerSampler,
    prompt: str,
    max_tokens_generated=100,
    kv_cache: KeyValueCache | None = None,
    verbose=False,
    seed: int | None = None,
    **kwargs
) -> str:

    self.model.eval()
    input_ids = self.tokenizer.encode(prompt, return_tensors="pt").to(device)[0]
    if seed is not None:
        np.random.seed(seed)
        t.manual_seed(seed)

    for i in tqdm(range(max_tokens_generated)):
        # Get new logits (make sure we don't pass in more tokens than the model's context length)
        logits, kv_cache = self.model(input_ids[None, -self.cfg.n_ctx:], kv_cache)
        # We only take logits for the last token, because this is what we're sampling
        logits = logits[0, -1]
        # Get next token (as a tensor of size (1, 1) so we can concat it to input_ids)
        next_token = t.tensor([TransformerSampler.sample_next_token(input_ids, logits, **kwargs)], device=device)
        # Create new input ids string, with shape (1, old_seq_len + 1)
        input_ids = t.cat([input_ids, next_token], dim=-1)
        # Print out results, if required
        if verbose:
            print(self.tokenizer.decode(input_ids), end="\r")
        # If our new token was the end-of-text token, stop
        if next_token == getattr(self.tokenizer, "eos_token_id", None):
            break

    return self.tokenizer.decode(input_ids)


TransformerSampler.sample = sample_with_cache
```
</details>

<details>
<summary>cache 버전과 no-cache 버전이 동일한 출력을 생성하는지 확인하고 (그리고 속도를 비교하기 위한) 코드입니다</summary>

```python
device = t.device("cuda") # can also try "cpu"

model = DemoTransformer(Config()).to(device)
model.load_state_dict(reference_gpt2.state_dict(), strict=False);

initial_text = "Eliezer Shlomo Yudkowsky (born September 11, 1979) is an American decision and artificial intelligence (AI) theorist and writer, best known for"
# input_ids = tokenizer.encode(initial_text, return_tensors="pt").squeeze()

sampler = TransformerSampler(model, tokenizer)

# Run the noncached version
t0 = time.time()
text = sampler.sample(
    initial_text,
    temperature=0.7,
    top_p=0.95,
    seed=0,
)
print(f"Time taken (without cache): {time.time() - t0:.2f} seconds")
rprint(f"Model output:\n\n[bold dark_orange]{text}[/]")

# Run the cached version
t0 = time.time()
text_with_cache = sampler.sample(
    initial_text,
    temperature=0.7,
    top_p=0.95,
    seed=0,
    kv_cache=KeyValueCache.new_empty(sampler.cfg)
)
print(f"Time taken (with cache): {time.time() - t0:.2f} seconds")
rprint(f"Model output:\n\n[bold dark_orange]{text_with_cache}[/]")

# # Check they are the same
assert text == text_with_cache, "Your outputs are different, meaning you've probably made a mistake in your cache implementation (or failed to use random seeds)."
print("Tests passed!")
```

</details>

cache 구현이 어느 정도의 속도 향상을 제공하는 것을 확인할 수 있겠지만, 매 단계에서 모든 token이 아닌 단 하나의 추가 token만 계산한다는 점에서 기대할 수 있는 `seq_len`배의 속도 향상에는 아마 미치지 못할 것입니다. 왜 그럴까요? 답은 딥러닝의 계산 및 메모리 비용과 관련된 모든 것이 그렇듯, 그렇게 간단하지 않기 때문입니다. 모델의 forward pass 속도에 병목 현상을 일으킬 수 있는 수많은 다양한 요인들이 존재합니다. 만약 이를 CPU에서 시도하신다면, 훨씬 더 눈에 띄는 속도 향상을 얻으실 수 있을 것입니다.

이러한 주제들에 대해 더 자세히 알아보시려면 [here](https://kipp.ly/blog/transformer-inference-arithmetic/#kv-cache)을(를) 참조하시기 바랍니다.

## 보너스 - 캐싱된 빔 서치 (cached beam search)

빔 서치 함수가 캐싱을 사용하도록 수정할 수 있습니까?

이전에 캐시를 어떻게 구현했느냐에 따라, 빔 서치에는 다른 형태의 캐싱이 더 적합하다는 것을 알게 될 수도 있습니다.

마찬가지로, 아래 드롭다운에 예시 구현을 제공했습니다. 이는 위의 캐시 구현과 `beam_search`의 이전 솔루션을 기반으로 합니다.

<details>
<summary>캐싱된 빔 서치 함수</summary>

앞서 언급했듯이, 모듈화된 코드 덕분에 캐시 지원을 추가할 때 변경해야 할 부분이 많지 않습니다.

```python
@dataclass
class Beams:
    '''Class to store beams during beam search.'''
    model: DemoTransformer
    tokenizer: GPT2TokenizerFast
    logprob_sums: Float[Tensor, "batch"]
    tokens: Int[Tensor, "batch seq"]
    kv_cache: KeyValueCache | None = None

    def __getitem__(self, idx) -> "Beams":
        '''Helpful function allowing you to take a slice of the beams object along the batch dimension.'''
        return Beams(
            self.model,
            self.tokenizer,
            self.logprob_sums[idx],
            self.tokens[idx],
            self.kv_cache[:, :, idx] if self.kv_cache is not None else None
        )

    @property
    def logprobs_and_completions(self) -> list[tuple[float, str]]:
        '''Returns self as a list of logprob sums and completions (useful for getting final output).'''
        return [
            (logprob_sum.item(), self.tokenizer.decode(tokens))
            for (logprob_sum, tokens) in zip(self.logprob_sums, self.tokens)
        ]


    def generate(self, k: int, no_repeat_ngram_size: int | None = None) -> "Beams":
        '''
        Starting from the current set of beams (i.e. self.tokens) and returns a new set of `len(self.tokens) * k` beams,
        containing the best `k` continuations for each of the original beams.

        Optional argument `no_repeat_ngram_size` means your model won't generate any sequences with a repeating n-gram
        of this length. 
        '''
        # Get the output logprobs for the next token (for every sequence in current beams)
        logprobs, kv_cache = self.model(self.tokens, self.kv_cache)
        logprobs = logprobs[:, -1, :].log_softmax(-1)

        # Get the top `toks_per_beam` tokens for each sequence
        topk_logprobs, topk_tokenIDs = self.get_topk_non_repeating(logprobs, no_repeat_ngram_size, k=k)

        # Add new logprobs & concat new tokens. When doing this, we need to add an extra `k` dimension since our current
        # logprobs & tokens have shape (batch,) and (batch, seq), but our new ones both have shape (batch, k)
        new_logprob_sums = einops.repeat(self.logprob_sums, "b -> b k", k=k) + topk_logprobs
        new_tokens = t.concat([einops.repeat(self.tokens, "b s -> b k s", k=k), topk_tokenIDs.unsqueeze(-1)], dim=-1)

        return Beams(self.model, self.tokenizer, new_logprob_sums.flatten(), new_tokens.flatten(0, 1), new_kv_cache)


    def filter(self, k: int) -> tuple["Beams", "Beams"]:
        '''
        Returns:
            best_beams: Beams
                filtered version of self, containing all best `k` which are also not terminated.
            early_terminations: Beams
                filtered version of self, containing all best `k` which are also terminated.
        '''
        # Get the indices of top `k` beams
        top_beam_indices = self.logprob_sums.topk(k=k, dim=0).indices.tolist()
        # Get the indices of terminated sequences
        new_tokens = self.tokens[:, -1]
        terminated_indices = t.nonzero(new_tokens == self.tokenizer.eos_token_id)

        # Get the indices of the `k` best sequences (some terminated, some not terminated)
        best_continuing = [i for i in top_beam_indices if i not in terminated_indices]
        best_terminated = [i for i in top_beam_indices if i in terminated_indices]

        # Return the beam objects from these indices
        return self[best_continuing], self[best_terminated]


    def get_topk_non_repeating(
        self,
        logprobs: Float[Tensor, "batch d_vocab"],
        no_repeat_ngram_size: int | None,
        k: int,
    ) -> tuple[Float[Tensor, "k"], Int[Tensor, "k"]]:
        """
        logprobs:
            tensor of the log-probs for the next token
        no_repeat_ngram_size:
            size of ngram to avoid repeating
        k:
            number of top logits to return, for each beam in our collection

        Returns:
            equivalent to the output of `logprobs.topk(dim=-1)`, but makes sure that no returned tokens would produce an
            ngram of size  `no_repeat_ngram_size` which has already appeared in `self.tokens`.
        """
        batch, seq_len = self.tokens.shape

        # If completion isn't long enough for a repetition, or we have no restrictions, just return topk
        if (no_repeat_ngram_size is not None) and (seq_len > no_repeat_ngram_size - 1):
            # Otherwise, we need to check for ngram repetitions
            # First, get the most recent `no_repeat_ngram_size-1` tokens
            last_ngram_prefix = self.tokens[:, seq_len - (no_repeat_ngram_size - 1) :]
            # Next, find all the tokens we're not allowed to generate, by checking all past ngrams for a match
            for i in range(seq_len - (no_repeat_ngram_size - 1)):
                ngrams = self.tokens[:, i : i + no_repeat_ngram_size]  # (batch, ngram)
                ngrams_are_repeated = (ngrams[:, :-1] == last_ngram_prefix).all(-1)  # (batch,)
                ngram_end_tokens = ngrams[:, [-1]]  # (batch, 1)
                # Fill logprobs with neginf wherever the ngrams are repeated
                logprobs[range(batch), ngram_end_tokens] = t.where(
                    ngrams_are_repeated, -1.0e4, logprobs[range(batch), ngram_end_tokens]
                )

        # Finally, get our actual tokens
        return logprobs.topk(k=k, dim=-1)

    def print(self, title="Best completions", max_print_chars=80) -> None:
        '''
        Prints out a set of sequences with their corresponding logitsums.
        '''
        if len(self.tokens) == 0:
            return
        table = Table("logitsum", "completion", title=title)
        for logprob_sum, tokens in zip(self.logprob_sums, self.tokens):
            text = self.tokenizer.decode(tokens)
            if len(repr(text)) > max_print_chars:
                text = text[:int(0.3 * max_print_chars)] + " ... " + text[-int(0.7 * max_print_chars):]
            table.add_row(f"{logprob_sum:>8.3f}", repr(text))
        rprint(table)


    @t.inference_mode()
    def beam_search(
        self,
        prompt: str,
        num_return_sequences: int,
        num_beams: int,
        max_new_tokens: int,
        no_repeat_ngram_size: int | None = None,
        kv_cache: KeyValueCache | None = None,
    ) -> list[tuple[float, Tensor]]:
        '''
        Implements a beam search, by repeatedly performing the `generate` and `filter` steps (starting from the initial
        prompt) until either of the two stopping criteria are met: (1) we've generated `max_new_tokens` tokens, or (2)
        we've generated `num_returns_sequences` terminating sequences.
        '''
        assert num_return_sequences <= num_beams
        self.model.eval()

        tokens = self.tokenizer.encode(prompt, return_tensors="pt").to(device)

        final_logprobs_and_completions = []  # we add to this list as we get terminated beams
        best_beams = Beams(self.model, self.tokenizer, t.tensor([0.0]).to(device), tokens)  # start with just 1 beam

        for _ in tqdm(range(max_new_tokens)):
            # Generate & filter beams
            best_beams = best_beams.generate(k=num_beams, no_repeat_ngram_size=no_repeat_ngram_size)
            best_beams, best_beams_terminated = best_beams.filter(k=num_beams)

            # Add terminated beams to our list, and return early if we have enough
            final_logprobs_and_completions.extend(best_beams_terminated.logprobs_and_completions)
            if len(final_logprobs_and_completions) >= num_return_sequences:
                return final_logprobs_and_completions[:num_return_sequences]

        # Return terminated beams plus the best ongoing beams of length `orig_len + max_new_tokens`
        final_logprobs_and_completions.extend(best_beams.logprobs_and_completions)
        return final_logprobs_and_completions[:num_return_sequences]


```

</details>

<details>
<summary>캐시 버전과 비캐시 버전이 동일한 출력을 생성하는지 확인하고 (속도를 비교하기 위한) 코드</summary>

```python
prompt = "For you, the day Bison graced your village was the most important day of your life. But for me, it was"
orig_len = len(tokenizer.encode(prompt))

beam_search_kwargs = dict(
    prompt=prompt,
    num_return_sequences=3,
    num_beams=20,
    max_new_tokens=60,
    no_repeat_ngram_size=2,
    verbose=False
)

sampler = TransformerSampler(model, tokenizer)

# Run the noncached version
t0 = time.time()
final_logitsums_and_completions = sampler.beam_search(**beam_search_kwargs)
logprob_sum, text = final_logitsums_and_completions[0]
avg_logprob_as_prob = t.tensor(logprob_sum / (len(tokenizer.encode(text)) - orig_len)).exp().item()
print(f"Time (without cache): {time.time() - t0:.2f} seconds")
print(f"Avg logprob (expressed as a probability) = {avg_logprob_as_prob:.3f}")
rprint(f"Output:\n\n[bold dark_orange]{text}[/]\n\n")

# Run the cached version
t0 = time.time()
beam_search_kwargs["kv_cache"] = KeyValueCache.new_empty(model.cfg)
final_logitsums_and_completions = sampler.beam_search(**beam_search_kwargs)
logprob_sum, text_with_cache = final_logitsums_and_completions[0]
avg_logprob_as_prob = t.tensor(logprob_sum / (len(tokenizer.encode(text)) - orig_len)).exp().item()
print(f"Time (with cache): {time.time() - t0:.2f} seconds")
print(f"Avg logprob (as probability) = {avg_logprob_as_prob:.3f}", end="")
rprint(f"Output:\n\n[bold dark_orange]{text_with_cache}[/]\n\n")

# Check they are the same
assert text == text_with_cache, "Your outputs are different, meaning you've probably made a mistake in your cache implementation."
print("Tests passed!")
```

</details>